# Ball detection to 80%: items 1-3 on Kaggle

Runs `docs/plans/BALL_IMPROVEMENT_PLAN.md` items 1-3 end to end: **miss diagnosis**, the **ball
tracker** (tuned on `select`, gated on `test`), and the **c5d specialist** (dataset, 2-GPU training
warm-started from c5c, gates, ONNX export). Every stage streams its log into the cell, writes
`/kaggle/working/ball_improvement/logs/<stage>.log`, and is resumable: re-running a cell skips
finished work. The last cells collect everything into `/kaggle/working/deliverables/` and
`deliverables.zip`.

**Settings:** Accelerator **GPU T4 x2**, Internet **on** (pip).

**Inputs** (Add Input):

| input | what | used for |
|---|---|---|
| `sabihalii/soccernetgsr` | SoccerNet-GSR frames + labels (`train/`, `valid/`, optionally GSR `test/`) | everything |
| `sabihalii/ball-detector-scripts` | its `best.pt`: the main athlete/ball detector | main cache |
| a dataset holding **`c5c_best.pt`** | `runs/ball_temporal/c5c/best.pt` from the repo, renamed | diagnosis, tracker, c5d warm start |
| (optional) an earlier run's output | `main_cache.zip` or `ball_improvement/` | skip the cache / resume training |

The c5c checkpoint is not on Kaggle yet: upload `runs/ball_temporal/c5c/best.pt` (58 MB) as a new
version of `ball-detector-scripts` (or a new dataset) under the name `c5c_best.pt`. The runner finds it
anywhere under `/kaggle/input`.

**Time** (one 12 h session, 2xT4): setup + cache ~1 h, diagnosis ~20 min, tracker ~1-2 h, c5d dataset
~1 h, training gets what's left minus 2.5 h, eval + export + deliverables ~2 h. Training stops itself
before the session ends; see **Resume** at the bottom.

**Disk:** `/kaggle/working` keeps ~20 GB of saved output. The data stage copies the GSR dataset there
only if it fits next to the caches and the crop dataset (`--copy auto`); otherwise it reads it in place
from `/kaggle/input` and says so. Run **Cleanup** before Save Version.

**Run the cells top to bottom.** For a ~15-minute dry run first, set `SMOKE = True` in the next cell.

Generated 2026-10-25 15:37 by `scripts/kaggle/build_ball_notebook.py` from commit `36aff33 + uncommitted changes`.

| kit file | sha256 |
|---|---|
| `scripts/ball_temporal/gsr_io.py` | `65c26621a3f8` |
| `scripts/ball_temporal/cache_main_detections.py` | `1494444bd593` |
| `scripts/ball_temporal/camera_motion_cache.py` | `fb2be1be8ebc` |
| `scripts/ball_temporal/heatmap_model.py` | `cadf23b723de` |
| `scripts/ball_temporal/build_ball_temporal_dataset.py` | `fa53456d45cf` |
| `scripts/ball_temporal/train_ball_temporal.py` | `ed27c0f27510` |
| `scripts/ball_temporal/export_ball_specialist.py` | `339b99ca24a2` |
| `scripts/eval/ball_det/eval_ball_temporal.py` | `e96c498ca573` |
| `scripts/eval/ball_det/ball_system.py` | `770eb069ecd3` |
| `scripts/eval/ball_det/diagnose_ball_misses.py` | `acf5b480b3a3` |
| `gsr_pipeline/ball_tracker.py` | `4c49fc31306b` |
| `notebooks/kaggle/ball_pipeline.py` | `d5ca9f5b03f0` |

In [ ]:
# Settings. Edit paths here if your inputs are named differently; leave None to auto-detect.
import os, subprocess, sys
from pathlib import Path

SMOKE = False                       # True: 1 sequence per split, 64 frames, a 256-sample epoch -> ~15 min
GSR_INPUT = "/kaggle/input/datasets/sabihalii/soccernetgsr"
MAIN_WEIGHTS = None                 # e.g. "/kaggle/input/ball-detector-scripts/best.pt"
C5C_CKPT = None                     # e.g. "/kaggle/input/ball-detector-scripts/c5c_best.pt"
MAIN_CACHE_ZIP = None               # e.g. "/kaggle/input/<earlier-run>/deliverables/main_cache.zip"
EPOCHS, LR, BATCH = 20, 1.5e-4, 32  # c5d training (warm start from c5c; batch is per GPU)

KIT = "/kaggle/working/kit/cv_service"
WORK = "/kaggle/working/" + ("ball_smoke" if SMOKE else "ball_improvement")
BASE = ["--gsr_input", GSR_INPUT, "--work", WORK, "--epochs", str(1 if SMOKE else EPOCHS),
        "--lr", str(LR), "--batch", str(BATCH)]
for flag, val in (("--main_weights", MAIN_WEIGHTS), ("--c5c_ckpt", C5C_CKPT), ("--main_cache_zip", MAIN_CACHE_ZIP)):
    if val:
        BASE += [flag, val]
if SMOKE:
    BASE += ["--max_seqs", "1", "--max_frames", "64", "--limit", "256", "--search_passes", "1", "--copy", "never"]


def RUN(stages, *extra):
    """Run pipeline stages; every log line appears here as it is printed."""
    cmd = [sys.executable, "-u", f"{KIT}/notebooks/kaggle/ball_pipeline.py", "--stages", stages, *BASE, *extra]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"stages {stages} stopped (exit {p.returncode}); fix the cause above and re-run this cell")


def SHOW(rel):
    """Render a report from the run folder."""
    from IPython.display import Markdown, display
    path = Path(WORK) / rel
    display(Markdown(path.read_text(encoding="utf-8")) if path.exists() else Markdown(f"*{path} not written yet*"))


print("kit:", KIT, "| work:", WORK, "| smoke:", SMOKE)

## 0. Kit files

These cells write the code to `/kaggle/working/kit/cv_service`. Run them all; don't edit them here (edit the repo and regenerate the notebook).

In [ ]:
import os
for d in ['gsr_pipeline', 'notebooks/kaggle', 'scripts/ball_temporal', 'scripts/eval/ball_det']:
    os.makedirs(f'/kaggle/working/kit/cv_service/{d}', exist_ok=True)
print('kit folders ready')

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_temporal/gsr_io.py
"""
gsr_io.py -- SoccerNet-GSR readers and the sequence splits for the temporal ball specialist
(docs/path_b/plans/BALL_TEMPORAL_PLAN.md §3, §6).

Standalone (numpy, opencv-python only), so the scripts in this folder run on Kaggle as is.

Splits, by sequence (decision A, 2026-10-12):
  train   all GSR `train` sequences. The main detector was trained on them, so their frames give
          easy positives and negatives only, never hard examples.
  pool    `valid` sequences outside test and select, plus every GSR `test` sequence (added
          2026-10-15). The main detector never trained on either: the source of hard examples.
  select  10 `valid` sequences, evenly spaced through the non-test ones: model selection and
          early stopping.
  test    the 15 held-out `valid` sequences: final gates only.

Sources: a directory searched recursively for sequence folders (Labels-GameState.json + img1/),
or failing that the split zips (train.zip / valid.zip), read without extracting.
"""

from __future__ import annotations

import json
import zipfile
from dataclasses import dataclass
from pathlib import Path, PurePosixPath

import numpy as np

TEST_SEQS = frozenset({21, 22, 23, 26, 30, 33, 36, 40, 44, 47, 50, 54, 80, 86, 90})
N_SELECT = 10
BALL_CAT = 4
PERSON_CATS = frozenset({1, 2, 3, 7})


@dataclass(frozen=True)
class SeqRef:
    name: str          # SNGS-060
    src_split: str     # train | valid | gsrtest (the GSR folder it came from)
    kind: str          # dir | zip
    container: str     # sequence dir, or zip path
    prefix: str        # inside-zip prefix of the sequence folder ("" for dir)
    split: str = ""    # train | pool | select | test, filled in by discover()

    @property
    def number(self) -> int:
        return int(self.name.split("-")[-1])


def _gsr_split_of(parts) -> str | None:
    low = [p.lower() for p in parts]
    if any("train" in p for p in low):
        return "train"
    if any("valid" in p for p in low):
        return "valid"
    if any(p == "test" or p.startswith("test.") for p in low):
        return "gsrtest"            # the GSR test split; not our held-out `test` (valid sequences)
    return None


def select_seqs(valid_numbers) -> set[int]:
    """N_SELECT sequences evenly spaced through the sorted non-test valid sequences."""
    rest = sorted(n for n in valid_numbers if n not in TEST_SEQS)
    if len(rest) <= N_SELECT:
        return set(rest)
    return {rest[int((i + 0.5) * len(rest) / N_SELECT)] for i in range(N_SELECT)}


def discover(src: Path) -> list[SeqRef]:
    """Sequence folders under src, else sequences inside zips under src. One entry per name."""
    found: dict[str, SeqRef] = {}
    for lab in sorted(Path(src).rglob("Labels-GameState.json")):
        seq = lab.parent
        gsr = _gsr_split_of(seq.relative_to(src).parts)
        if gsr and seq.name not in found and (seq / "img1").is_dir():
            found[seq.name] = SeqRef(seq.name, gsr, "dir", str(seq), "")
    if not found:
        for zp in sorted(Path(src).rglob("*.zip")):
            with zipfile.ZipFile(zp) as z:
                for n in z.namelist():
                    if not n.endswith("Labels-GameState.json"):
                        continue
                    pre = str(PurePosixPath(n).parent)
                    name = PurePosixPath(pre).name
                    gsr = _gsr_split_of(PurePosixPath(pre).parts) or _gsr_split_of([zp.name])
                    if gsr and name.startswith("SNGS") and name not in found:
                        found[name] = SeqRef(name, gsr, "zip", str(zp), pre)
    sel = select_seqs(r.number for r in found.values() if r.src_split == "valid")

    def split_of(r: SeqRef) -> str:
        if r.src_split == "train":
            return "train"
        if r.src_split == "gsrtest":
            return "pool"
        return "test" if r.number in TEST_SEQS else "select" if r.number in sel else "pool"

    refs = [SeqRef(r.name, r.src_split, r.kind, r.container, r.prefix, split_of(r)) for r in found.values()]
    return sorted(refs, key=lambda s: s.number)


class SeqReader:
    """Reads one sequence's labels and frames from a folder or a zip (one per process)."""

    def __init__(self, ref: SeqRef):
        self.ref = ref
        self._zip = zipfile.ZipFile(ref.container) if ref.kind == "zip" else None

    def read(self, rel: str) -> bytes:
        if self._zip is not None:
            return self._zip.read(f"{self.ref.prefix}/{rel}")
        return (Path(self.ref.container) / rel).read_bytes()

    def labels(self) -> dict:
        return json.loads(self.read("Labels-GameState.json"))

    def close(self):
        if self._zip is not None:
            self._zip.close()


def decode(buf: bytes):
    """JPEG bytes -> BGR image. cv2.imdecode releases the GIL, so threads can share it."""
    import cv2
    img = cv2.imdecode(np.frombuffer(buf, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("cannot decode frame")
    return img


def parse_labels(d: dict):
    """-> im_dir, {frame: file_name}, {frame: [(cx,cy,w,h,track_id)]}, persons, line points, (W,H)."""
    info = d.get("info", {})
    im_dir = info.get("im_dir", "img1")
    W, H = 1920, 1080
    files, id2f = {}, {}
    for im in d["images"]:
        f = int(Path(im["file_name"]).stem)
        files[f] = im["file_name"]
        id2f[im["image_id"]] = f
        W, H = im.get("width", W), im.get("height", H)
    balls, persons, lines = {}, {}, {}
    for a in d["annotations"]:
        f = id2f.get(a["image_id"])
        if f is None:
            continue
        cat = a.get("category_id")
        if cat == BALL_CAT or cat in PERSON_CATS:
            b = a["bbox_image"]
            row = (float(b["x_center"]), float(b["y_center"]), float(b["w"]), float(b["h"]), a.get("track_id"))
            (balls if cat == BALL_CAT else persons).setdefault(f, []).append(row)
        elif a.get("supercategory") == "pitch":
            pts = [(p["x"] * W, p["y"] * H) for seg in a.get("lines", {}).values() for p in seg]
            lines[f] = [(x, y) for x, y in pts if 0 <= x < W and 0 <= y < H]
    return im_dir, files, balls, persons, lines, (W, H)

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_temporal/cache_main_detections.py
"""
cache_main_detections.py -- step 1 of docs/path_b/plans/BALL_TEMPORAL_PLAN.md §9: run the main
athlete/ball detector over every frame and cache its ball detections per sequence.

The cache feeds step 2 (the temporal dataset builder): it marks hard frames (GSR has a ball,
the main detector missed it) and centres crops on the last prediction, as the runtime does.

Standalone apart from gsr_io.py (same folder). Needs ultralytics>=8.4.0 (YOLO26).

  list     print the sequences per split (train / pool / select / test); no GPU
  run      detect and cache; --shard i --nshards n splits the sequences across GPUs
  summary  main-detector recall per split and hard-frame counts, from the cache; no GPU

Cache: <out>/<split>/<SNGS-xxx>.json =
  {"meta": {weights, imgsz, conf, top}, "frames": {"<frame>": {"balls": [[cx, cy, w, h, conf], ...],
                                                               "gt": [[cx, cy, size], ...]}}}
balls: every ball detection with conf >= --conf, highest first, at most --top, in native pixels.
Finished sequences are skipped, and each file is written atomically, so a stopped run resumes.

    python cache_main_detections.py run --src /kaggle/input/datasets/sabihalii/soccernetgsr \
        --out /kaggle/working/main_cache --weights /kaggle/working/kit/best.pt --device 0 --half
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from gsr_io import SeqReader, decode, discover, parse_labels  # noqa: E402

SPLITS = ("train", "pool", "select", "test")
MATCH_PX = 15.0      # ball match radius, as the gates (plan §8)
HARD_CONF = 0.15     # a frame is hard when no ball at this confidence is within MATCH_PX of GT


def _balls(result, cls: int, top: int) -> list[list[float]]:
    b = result.boxes
    if b is None or len(b) == 0:
        return []
    keep = b.cls.cpu().numpy().astype(int) == cls
    xywh, conf = b.xywh.cpu().numpy()[keep], b.conf.cpu().numpy()[keep]
    order = np.argsort(-conf)[:top]
    return [[round(float(v), 2) for v in (*xywh[i], conf[i])] for i in order]


def _refs(a):
    refs = [r for r in discover(a.src) if r.split in a.splits]
    if a.seqs:
        refs = [r for r in refs if r.name in a.seqs]
    return refs


def stage_list(a):
    refs = discover(a.src)
    for s in SPLITS:
        names = [r.name for r in refs if r.split == s]
        print(f"{s:6s} {len(names):3d}  {' '.join(n.replace('SNGS-', '') for n in names)}")


def stage_run(a):
    from ultralytics import YOLO
    model = YOLO(a.weights)
    names = {v: k for k, v in model.names.items()}
    if "ball" not in names:
        raise SystemExit(f"model has no 'ball' class: {model.names}")
    cls = names["ball"]
    kw = dict(imgsz=a.imgsz, conf=a.conf, iou=0.7, max_det=300, device=a.device, verbose=False)
    if a.half:
        kw["half"] = True                         # only when asked: Ultralytics 8.4 warns on any `half`
    meta = {"weights": str(a.weights), "imgsz": a.imgsz, "conf": a.conf, "top": a.top}

    refs = [r for i, r in enumerate(_refs(a)) if i % a.nshards == a.shard]
    tag = f"[gpu{a.device}]"
    print(f"{tag} {len(refs)} sequences", flush=True)
    pool = ThreadPoolExecutor(a.decode_threads)
    t_all = time.time()
    for k, ref in enumerate(refs, 1):
        out = a.out / ref.split / f"{ref.name}.json"
        if out.exists():
            print(f"{tag} {k}/{len(refs)} {ref.name}: cached", flush=True)
            continue
        out.parent.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        rd = SeqReader(ref)
        im_dir, files, gt, *_ = parse_labels(rd.labels())
        frames = sorted(files)[:a.max_frames] if a.max_frames else sorted(files)
        rec = {}
        for i in range(0, len(frames), a.batch):
            chunk = frames[i:i + a.batch]
            bufs = [rd.read(f"{im_dir}/{files[f]}") for f in chunk]   # zip reads stay on one thread
            imgs = list(pool.map(decode, bufs))
            for f, r in zip(chunk, model.predict(imgs, **kw)):
                rec[str(f)] = {"balls": _balls(r, cls, a.top),
                               "gt": [[round(b[0], 2), round(b[1], 2), round(max(b[2], b[3]), 2)]
                                      for b in gt.get(f, [])]}
        rd.close()
        tmp = out.with_suffix(".tmp")
        tmp.write_text(json.dumps({"meta": meta, "frames": rec}))
        tmp.replace(out)
        dt = time.time() - t0
        print(f"{tag} {k}/{len(refs)} {ref.split}/{ref.name}: {len(rec)} frames in {dt:.0f}s "
              f"({1000 * dt / max(len(rec), 1):.0f} ms/frame), elapsed {(time.time() - t_all) / 60:.1f} min",
              flush=True)
    print(f"{tag} done in {(time.time() - t_all) / 60:.1f} min", flush=True)


def _hit(balls, gts, conf) -> bool:
    return any(b[4] >= conf and np.hypot(b[0] - g[0], b[1] - g[1]) <= MATCH_PX for b in balls for g in gts)


def stage_summary(a):
    lines = ["# Main-detector cache summary", "",
             f"Hard frame: GSR has a ball and no main-detector ball at conf >= {HARD_CONF} is within "
             f"{MATCH_PX:.0f} px of it.", "",
             "| split | sequences | frames | frames with GT ball | main recall (top ball) | hard frames | hard share |",
             "|---|---|---|---|---|---|---|"]
    for s in SPLITS:
        files = sorted((a.out / s).glob("SNGS-*.json"))
        if not files:
            continue
        n_fr = n_gt = n_top = n_hard = 0
        for fp in files:
            for v in json.loads(fp.read_text())["frames"].values():
                n_fr += 1
                if not v["gt"]:
                    continue
                n_gt += 1
                n_top += _hit(v["balls"][:1], v["gt"], HARD_CONF)
                n_hard += not _hit(v["balls"], v["gt"], HARD_CONF)
        lines.append(f"| {s} | {len(files)} | {n_fr} | {n_gt} | {n_top / max(n_gt, 1):.2%} | "
                     f"{n_hard} | {n_hard / max(n_gt, 1):.2%} |")
    lines += ["", "If train recall is clearly above pool/select/test, the detector memorised train, "
                  "and hard examples must come from pool only (plan §6, decision A)."]
    text = "\n".join(lines)
    print(text)
    (a.out / "summary.md").write_text(text)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["list", "run", "summary"])
    ap.add_argument("--src", type=Path, help="GSR source: sequence dirs or the split zips")
    ap.add_argument("--out", type=Path, default=Path("/kaggle/working/main_cache"))
    ap.add_argument("--weights", default="/kaggle/working/kit/best.pt", help="main athlete/ball detector")
    ap.add_argument("--splits", nargs="+", default=list(SPLITS), choices=SPLITS)
    ap.add_argument("--seqs", nargs="*", default=[], help="only these sequences, e.g. SNGS-060 (smoke tests)")
    ap.add_argument("--imgsz", type=int, default=1280, help="1280 = the 736x1280 runtime input after letterbox")
    ap.add_argument("--conf", type=float, default=0.05, help="lowest ball confidence kept in the cache")
    ap.add_argument("--top", type=int, default=5, help="most ball detections kept per frame")
    ap.add_argument("--device", default="0")
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--half", action="store_true", help="FP16 inference (GPU only)")
    ap.add_argument("--decode_threads", type=int, default=2, help="JPEG decode threads per process")
    ap.add_argument("--max_frames", type=int, default=0, help="first N frames per sequence (smoke tests)")
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--nshards", type=int, default=1)
    a = ap.parse_args()
    if a.stage in ("list", "run") and not a.src:
        ap.error("--src is required")
    {"list": stage_list, "run": stage_run, "summary": stage_summary}[a.stage](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_temporal/camera_motion_cache.py
"""
camera_motion_cache.py -- per-frame camera motion for every sequence, cached, for the ball
tracker's offline evaluation and the dataset builder (docs/plans/BALL_IMPROVEMENT_PLAN.md §2.3,
§3.2). Standalone apart from gsr_io.py (same folder). Needs numpy, opencv-python.

For frame t: a 2x3 similarity transform (rotation, uniform scale, translation) taking frame t-1
pixels to frame t pixels, estimated from sparse optical flow on a 960-wide grey image with
people and the ball masked out (GT boxes, expanded 20%), then scaled back to full resolution.
None when fewer than --min_tracked features survive or RANSAC inliers fall below --min_inliers
(a cut, a close-up): the tracker then treats the frame as having no motion information.

GT boxes are used for the mask, so this is slightly cleaner than the runtime, which masks
detections. The runtime estimate is gsr_pipeline/camera_motion.py; this file is the offline
stand-in and does not import it, so it runs in the Kaggle kit unchanged.

    run      compute and cache: <out>/<split>/<SNGS-xxx>.json = {"meta": ..., "frames": {"<t>": [6 floats] | null}}
    summary  share of frames with an estimate, median |translation| per split

    python scripts/ball_temporal/camera_motion_cache.py run --src data/datasets/SoccerNetGS \
        --out data/outputs/ball_temporal/motion_cache --splits select test --workers 4
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time
from multiprocessing import Pool
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from gsr_io import SeqReader, decode, discover, parse_labels  # noqa: E402

SPLITS = ("train", "pool", "select", "test")
WORK_W = 960


def _mask(shape, boxes, scale):
    m = np.full(shape, 255, np.uint8)
    for cx, cy, w, h, _ in boxes:
        w2, h2 = 0.6 * w * scale, 0.6 * h * scale
        x0, y0 = int(max(cx * scale - w2, 0)), int(max(cy * scale - h2, 0))
        x1, y1 = int(cx * scale + w2) + 1, int(cy * scale + h2) + 1
        m[y0:y1, x0:x1] = 0
    return m


def estimate(prev_g, cur_g, mask, scale, min_tracked=40, min_inliers=0.5):
    """Grey images at work resolution -> full-resolution 2x3 affine (frame t-1 -> t) or None."""
    import cv2
    pts = cv2.goodFeaturesToTrack(prev_g, maxCorners=400, qualityLevel=0.01, minDistance=8, mask=mask)
    if pts is None or len(pts) < min_tracked:
        return None
    nxt, st, _ = cv2.calcOpticalFlowPyrLK(prev_g, cur_g, pts, None, winSize=(21, 21), maxLevel=3)
    good = st.reshape(-1) == 1
    if good.sum() < min_tracked:
        return None
    a, inl = cv2.estimateAffinePartial2D(pts[good], nxt[good], method=cv2.RANSAC, ransacReprojThreshold=2.0)
    if a is None or inl is None or inl.mean() < min_inliers:
        return None
    a = a.copy()
    a[:, 2] /= scale                      # translation back to full-resolution pixels
    return a


def process_sequence(job):
    import cv2
    ref, out_dir, min_tracked, min_inliers = job
    out = Path(out_dir) / ref.split / f"{ref.name}.json"
    if out.exists():
        return ref.name, None, True
    reader = SeqReader(ref)
    try:
        im_dir, files, balls, persons, _lines, (W, H) = parse_labels(reader.labels())
        scale = WORK_W / W
        size = (WORK_W, int(round(H * scale)))
        rec, prev, ok = {}, None, 0
        for t in sorted(files):
            g = cv2.cvtColor(cv2.resize(decode(reader.read(f"{im_dir}/{files[t]}")), size,
                                        interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY)
            a = None
            if prev is not None and prev[0] == t - 1:
                m = _mask(prev[1].shape, persons.get(t - 1, []) + balls.get(t - 1, []), scale)
                a = estimate(prev[1], g, m, scale, min_tracked, min_inliers)
            rec[str(t)] = None if a is None else [round(float(v), 6) for v in a.reshape(-1)]
            ok += a is not None
            prev = (t, g)
        out.parent.mkdir(parents=True, exist_ok=True)
        tmp = out.with_suffix(".tmp")
        tmp.write_text(json.dumps({"meta": {"work_w": WORK_W, "min_tracked": min_tracked,
                                            "min_inliers": min_inliers}, "frames": rec}))
        tmp.replace(out)
        return ref.name, ok / max(len(files), 1), False
    finally:
        reader.close()


def load(cache_dir: Path, ref) -> dict:
    """{frame: 2x3 affine (np.ndarray) or None} for one sequence."""
    d = json.loads((Path(cache_dir) / ref.split / f"{ref.name}.json").read_text())["frames"]
    return {int(k): (None if v is None else np.array(v, float).reshape(2, 3)) for k, v in d.items()}


def stage_run(a):
    refs = [r for r in discover(a.src) if r.split in a.splits]
    if a.seqs:
        refs = [r for r in refs if r.name in a.seqs]
    print(f"camera motion: {len(refs)} sequences, {a.workers} workers -> {a.out}", flush=True)
    t0 = time.time()
    jobs = [(r, str(a.out), a.min_tracked, a.min_inliers) for r in refs]
    with Pool(a.workers) as pool:
        for i, (name, share, cached) in enumerate(pool.imap_unordered(process_sequence, jobs), 1):
            tail = "cached" if cached else f"{share:.0%} frames with an estimate"
            print(f"[motion {i}/{len(refs)}] {name}: {tail} ({time.time() - t0:.0f}s)", flush=True)


def stage_summary(a):
    lines = ["| split | sequences | frames | with an estimate | median translation px |", "|---|---|---|---|---|"]
    for s in SPLITS:
        files = sorted((a.out / s).glob("SNGS-*.json"))
        if not files:
            continue
        n = k = 0
        tr = []
        for fp in files:
            for v in json.loads(fp.read_text())["frames"].values():
                n += 1
                if v is not None:
                    k += 1
                    tr.append(float(np.hypot(v[2], v[5])))
        lines.append(f"| {s} | {len(files)} | {n} | {k / max(n, 1):.1%} | {np.median(tr) if tr else 0:.1f} |")
    text = "\n".join(lines)
    print(text)
    (a.out / "summary.md").write_text(text + "\n")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["run", "summary"])
    ap.add_argument("--src", type=Path)
    ap.add_argument("--out", type=Path, default=Path("data/outputs/ball_temporal/motion_cache"))
    ap.add_argument("--splits", nargs="+", default=list(SPLITS), choices=SPLITS)
    ap.add_argument("--seqs", nargs="*", default=[])
    ap.add_argument("--workers", type=int, default=min(8, os.cpu_count() or 4))
    ap.add_argument("--min_tracked", type=int, default=40)
    ap.add_argument("--min_inliers", type=float, default=0.5)
    a = ap.parse_args()
    if a.stage == "run" and not a.src:
        ap.error("--src is required")
    {"run": stage_run, "summary": stage_summary}[a.stage](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_temporal/heatmap_model.py
"""
heatmap_model.py -- the temporal ball specialist (docs/path_b/plans/BALL_TEMPORAL_PLAN.md §4-§5):
model, input construction, targets, loss and decode. Shared by training, evaluation and export.

Input: 3 frames t-2, t-1, t of one 640x640 window, as uint8 RGB (B, 9, 640, 640) in that order.
  9 channels: the three RGB crops, ImageNet-normalised.
  5 channels: RGB at t (normalised) + grey(t) - grey(t-1) and grey(t-1) - grey(t-2), scaled by 1/0.226.
Output: heatmap logits (B, 1, 320, 320) and sub-pixel offset (B, 2, 320, 320), stride 2.
A ball at crop pixel (x, y) peaks at heatmap cell (floor(x/2), floor(y/2)) with offset (x/2, y/2) - cell.

Needs torch and torchvision.
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

STRIDE = 2
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
DIFF_SCALE = 1 / 0.226
PRIOR = 0.1                       # initial heatmap sigmoid output, CenterNet-style


def build_input(frames: torch.Tensor, channels: int) -> torch.Tensor:
    """uint8 (B, 9, H, W) frames t-2|t-1|t RGB -> float model input with `channels` (5 or 9) channels."""
    x = frames.float() / 255
    mean = x.new_tensor(MEAN).view(1, 3, 1, 1)
    std = x.new_tensor(STD).view(1, 3, 1, 1)
    f2, f1, f0 = x[:, 0:3], x[:, 3:6], x[:, 6:9]
    if channels == 9:
        return torch.cat([(f - mean) / std for f in (f2, f1, f0)], 1)
    if channels == 5:
        g = [f.mean(1, keepdim=True) for f in (f2, f1, f0)]
        return torch.cat([(f0 - mean) / std, (g[2] - g[1]) * DIFF_SCALE, (g[1] - g[0]) * DIFF_SCALE], 1)
    raise ValueError(f"channels must be 5 or 9, got {channels}")


class _Up(nn.Module):
    def __init__(self, c_in, c_skip, c_out):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(c_in + c_skip, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], 1))


class BallHeatmapNet(nn.Module):
    """ResNet-18 encoder (first conv widened to `channels`), U-Net decoder to a stride-2 heatmap + offset."""

    def __init__(self, channels: int = 9, pretrained: bool = True):
        super().__init__()
        from torchvision.models import ResNet18_Weights, resnet18
        r = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        w = r.conv1.weight.data                                   # (64, 3, 7, 7)
        conv1 = nn.Conv2d(channels, 64, 7, 2, 3, bias=False)
        with torch.no_grad():
            if channels == 9:      # each frame gets the RGB weights, /3 so activations keep their size
                conv1.weight.copy_(torch.cat([w, w, w], 1) / 3)
            else:                  # RGB at t as pretrained; diff channels see a grey image -> summed weights
                g = w.sum(1, keepdim=True)
                conv1.weight.copy_(torch.cat([w, g * 0.5, g * 0.5], 1))
        self.channels = channels
        self.stem = nn.Sequential(conv1, r.bn1, r.relu)            # stride 2, 64
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4   # 4, 8, 16, 32
        self.up4 = _Up(512, 256, 256)
        self.up3 = _Up(256, 128, 128)
        self.up2 = _Up(128, 64, 64)
        self.up1 = _Up(64, 64, 64)                                 # -> stride 2
        self.hm = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(64, 1, 1))
        self.off = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(64, 2, 1))
        nn.init.constant_(self.hm[-1].bias, -2.19)                 # sigmoid ~ PRIOR

    def forward(self, x):
        s2 = self.stem(x)
        s4 = self.l1(self.pool(s2))
        s8 = self.l2(s4)
        s16 = self.l3(s8)
        s32 = self.l4(s16)
        d = self.up1(self.up2(self.up3(self.up4(s32, s16), s8), s4), s2)
        return self.hm(d), self.off(d)


def make_targets(points: torch.Tensor, hm_size: int):
    """points (B, K, 3) crop px (x, y, size), rows with size <= 0 unused ->
    heatmap (B, 1, S, S), peak cell index (B, K), offset (B, K, 2), mask (B, K)."""
    B, K, _ = points.shape
    dev = points.device
    valid = points[..., 2] > 0
    c = points[..., :2] / STRIDE
    cell = c.floor().clamp(0, hm_size - 1)
    off = c - cell
    sigma = (points[..., 2] / STRIDE / 4).clamp(min=1.5)
    ys = torch.arange(hm_size, device=dev, dtype=torch.float32).view(1, 1, hm_size, 1)
    xs = torch.arange(hm_size, device=dev, dtype=torch.float32).view(1, 1, 1, hm_size)
    d2 = (xs - cell[..., 0, None, None]) ** 2 + (ys - cell[..., 1, None, None]) ** 2
    g = torch.exp(-d2 / (2 * sigma[..., None, None] ** 2)) * valid[..., None, None]
    hm = g.amax(1, keepdim=True) if K else torch.zeros(B, 1, hm_size, hm_size, device=dev)
    ind = (cell[..., 1] * hm_size + cell[..., 0]).long()
    return hm, ind, off, valid


def focal_loss(logits: torch.Tensor, gt: torch.Tensor) -> torch.Tensor:
    """CenterNet penalty-reduced focal loss, normalised by the number of peaks (min 1)."""
    p = logits.float().sigmoid().clamp(1e-4, 1 - 1e-4)
    pos = gt.eq(1).float()
    neg = 1 - pos
    pos_loss = torch.log(p) * (1 - p) ** 2 * pos
    neg_loss = torch.log(1 - p) * p ** 2 * (1 - gt) ** 4 * neg
    return -(pos_loss.sum() + neg_loss.sum()) / pos.sum().clamp(min=1)


def offset_loss(off: torch.Tensor, ind: torch.Tensor, off_gt: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    B, _, S, _ = off.shape
    pred = off.float().view(B, 2, S * S).gather(2, ind.clamp(0, S * S - 1)[:, None].expand(-1, 2, -1))
    pred = pred.permute(0, 2, 1)                                   # (B, K, 2)
    m = mask[..., None].float()
    return (F.l1_loss(pred, off_gt, reduction="none") * m).sum() / m.sum().clamp(min=1)


@torch.no_grad()
def decode(logits: torch.Tensor, off: torch.Tensor):
    """-> (B, 3): x, y in crop pixels and confidence of the single strongest peak."""
    B, _, S, _ = logits.shape
    p = logits.float().sigmoid().view(B, -1)
    conf, i = p.max(1)
    iy, ix = (i // S).float(), (i % S).float()
    o = off.float().view(B, 2, -1).gather(2, i[:, None, None].expand(-1, 2, 1))[..., 0]
    return torch.stack([(ix + o[:, 0]) * STRIDE, (iy + o[:, 1]) * STRIDE, conf], 1)

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_temporal/build_ball_temporal_dataset.py
"""
build_ball_temporal_dataset.py -- step 2 of docs/path_b/plans/BALL_TEMPORAL_PLAN.md §9: the
3-frame, point-labelled crop dataset for the temporal ball specialist (plan §4-§6).

Standalone apart from gsr_io.py (same folder). Needs numpy, opencv-python. Reads the main-detector
cache from step 1 (cache_main_detections.py).

  build    write the samples and manifest.csv; resumes per sequence
  stats    counts per split and kind, from manifest.csv
  preview  contact sheets of random samples with their labels drawn, for the spot-check by eye

Per ball-labelled frame t (frames without a ball label give no crops, plan §3):
  * Window: 640x640 native pixels, shift-clamped into the frame (never padded), centred like the
    runtime (plan §6.3): on the main detector's top ball (conf >= --pred_conf) in the most recent
    of frames t-1 .. t-15; failing that, on the GT ball in the most recent labelled frame of that
    range, +-48 px jitter; failing both, no positive crop.
  * Hard frame: GT ball, and no main ball at conf >= --pred_conf within 15 px of it. Marked in
    every split; hard *positives* are only sampled as such outside `train` (decision A).
  * Positives: every hard frame (repeat 3 in pool, 1 elsewhere); other frames every 3rd (t % 3 == 0).
    A ball cut by the window is labelled if its centre and >= 50% of its box are inside, otherwise
    the crop is dropped. If the window misses the ball entirely (centred on a false positive, or
    a long gap), the crop is kept as kind `outside`, with an empty label: the runtime sees it too.
  * Negatives, from positive-sampled frames only: hard (person or pitch-line point, +-64 px) for
    15% of crops and random for 5%; no ball in the window at t, t-1 or t-2.
  * --centre predicted (docs/plans/BALL_IMPROVEMENT_PLAN.md §4.3): the window is centred on the
    ball tracker's prediction instead (gsr_pipeline/ball_tracker.py replayed over main sightings,
    with camera motion from camera_motion_cache.py), falling back to the rule above when there is
    no live track; centre_source "kf". --gap2_repeat k multiplies the repeat of pool hard
    positives whose centre is >= 2 frames old. Defaults reproduce the c5c dataset.
  * Frames t-1, t-2 are cut with the window chosen at t. Before the sequence start they repeat
    frame t (n_prev < 2). The 5% "repeat t" training case is an augmentation in step 3, not here.

Output (<out>/):
  <split>/<SNGS-xxx>/<frame>_<kind>.jpg   one sample: crops t-2 | t-1 | t side by side, 640x1920
  manifest.csv                            one row per sample (columns: MANIFEST_FIELDS)
  parts/<SNGS-xxx>.csv                    per-sequence manifest, written last (= sequence done)
balls: "cx cy size;..." in crop pixels of frame t (size = max(w, h) of the GSR box), empty if none.
main_conf: positives, the highest main-detector confidence within 15 px of the GT ball (0 if none);
negatives and `outside`, the highest main-detector confidence inside the window (0 if none).

    python scripts/ball_temporal/build_ball_temporal_dataset.py build --src data/datasets/SoccerNetGS \
        --cache data/outputs/ball_temporal/main_cache --out data/outputs/ball_temporal/dataset --workers 8
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import sys
import time
from multiprocessing import Pool
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "gsr_pipeline"))     # ball_tracker.py, plain import
from gsr_io import SeqReader, decode, discover, parse_labels  # noqa: E402

SPLITS = ("train", "pool", "select", "test")
CROP = 640
LOOKBACK = 15            # frames searched back for the window centre (= max_miss_frames)
JITTER = 48              # on GT-centred windows only
EASY_STEP = 3            # easy positives: every 3rd frame
HARD_REPEAT = 3          # hard positives in pool, via the manifest's repeat column
NEG_HARD_SHARE = 0.15    # of all crops
NEG_RANDOM_SHARE = 0.05
MIN_VISIBLE = 0.5
MATCH_PX = 15.0
MANIFEST_FIELDS = ["split", "seq", "frame", "kind", "hard", "repeat", "centre_source", "centre_gap",
                   "main_conf", "x0", "y0", "clamped", "n_prev", "n_balls", "ball_size", "ball_dx",
                   "ball_dy", "balls", "file"]


# --------------------------------------------------------------------------- geometry

def clamp_window(cx: float, cy: float, W: int, H: int) -> tuple[int, int, bool]:
    """Top-left of a CROP x CROP window centred at (cx, cy), shifted (never padded) into the frame."""
    x0, y0 = int(round(cx - CROP / 2)), int(round(cy - CROP / 2))
    cx0 = min(max(x0, 0), max(W - CROP, 0))
    cy0 = min(max(y0, 0), max(H - CROP, 0))
    return cx0, cy0, (cx0, cy0) != (x0, y0)


def balls_in_window(balls, x0, y0):
    """-> (points, dropped). points: [(cx, cy, size)] in crop px; dropped if a ball is cut ambiguously."""
    pts = []
    for cx, cy, w, h, _ in balls:
        l, t, r, b = cx - w / 2 - x0, cy - h / 2 - y0, cx + w / 2 - x0, cy + h / 2 - y0
        inter = max(min(r, CROP) - max(l, 0), 0) * max(min(b, CROP) - max(t, 0), 0)
        if inter <= 0:
            continue
        centre_in = 0 <= cx - x0 < CROP and 0 <= cy - y0 < CROP
        if inter / max(w * h, 1e-6) < MIN_VISIBLE or not centre_in:
            return [], True
        pts.append((cx - x0, cy - y0, max(w, h)))
    return pts, False


def overlaps_any_ball(balls, x0, y0, margin=8):
    return any(cx + w / 2 + margin > x0 and cx - w / 2 - margin < x0 + CROP and
               cy + h / 2 + margin > y0 and cy - h / 2 - margin < y0 + CROP
               for cx, cy, w, h, _ in balls)


def _cut(img, x0, y0):
    c = img[y0:y0 + CROP, x0:x0 + CROP]
    if c.shape[:2] != (CROP, CROP):
        raise ValueError(f"frame {img.shape[:2]} smaller than {CROP}px")
    return c


# --------------------------------------------------------------------------- per sequence

def _matched_conf(dets, gts) -> float:
    return max((d[4] for d in dets for g in gts if np.hypot(d[0] - g[0], d[1] - g[1]) <= MATCH_PX),
               default=0.0)


def _conf_in_window(dets, x0, y0) -> float:
    return max((d[4] for d in dets if x0 <= d[0] < x0 + CROP and y0 <= d[1] < y0 + CROP), default=0.0)


def _centre(t, dets, balls, pred_conf, rng):
    """Runtime-like window centre for frame t -> (cx, cy, source, gap) or None."""
    for g in range(1, LOOKBACK + 1):
        d = [b for b in dets.get(t - g, []) if b[4] >= pred_conf]
        if d:
            return d[0][0], d[0][1], "pred", g
    for g in range(1, LOOKBACK + 1):
        if balls.get(t - g):
            b = balls[t - g][0]
            jx, jy = rng.uniform(-JITTER, JITTER, 2)
            return b[0] + jx, b[1] + jy, "gt", g
    return None


def _tracker_centres(files, dets, motion, pred_conf, tracker_cfg: dict | None) -> dict:
    """--centre predicted: {t: (cx, cy, gap)} = the ball tracker's prediction at t (before it sees
    frame t), replayed over main-detector sightings only (conf >= pred_conf), for frames 1..LOOKBACK
    after the last accepted sighting. Same tracker as the runtime (gsr_pipeline/ball_tracker.py)."""
    import ball_tracker as bt
    cfg = bt.TrackerConfig(**{**(tracker_cfg or {}), "min_conf_main": pred_conf, "init_conf": pred_conf})
    tr = bt.BallTracker(cfg)
    out = {}
    for t in sorted(files):
        pred = tr.predict(t, motion.get(t))
        gap = tr.gap if tr.last_meas is not None else None
        if pred is not None and gap is not None and 1 <= gap <= LOOKBACK:
            out[t] = (pred[0], pred[1], gap)
        tr.update([bt.BallCandidate(d[0], d[1], d[4]) for d in dets.get(t, []) if d[4] >= pred_conf])
    return out


def process_sequence(job):
    import cv2
    ref, cache_dir, out, seed, pred_conf, quality, centre_mode, motion_dir, gap2_repeat, tracker_cfg = job
    part = Path(out) / "parts" / f"{ref.name}.csv"
    if part.exists():
        with open(part, newline="") as f:
            return ref.name, list(csv.DictReader(f)), True
    cache = json.loads((Path(cache_dir) / ref.split / f"{ref.name}.json").read_text())["frames"]
    dets = {int(k): v["balls"] for k, v in cache.items()}       # highest confidence first
    reader = SeqReader(ref)
    try:
        im_dir, files, balls, persons, lines, (W, H) = parse_labels(reader.labels())
        kf = {}
        if centre_mode == "predicted":
            import camera_motion_cache as cmc
            motion = cmc.load(Path(motion_dir), ref) if motion_dir else {}
            kf = _tracker_centres(files, dets, motion, pred_conf, tracker_cfg)
        seq_dir = Path(out) / ref.split / ref.name
        seq_dir.mkdir(parents=True, exist_ok=True)
        frames_cache: dict[int, np.ndarray] = {}

        def frame(f):
            if f not in frames_cache:
                frames_cache[f] = decode(reader.read(f"{im_dir}/{files[f]}"))
            return frames_cache[f]

        rows = []
        for t in sorted(files):
            bt = balls.get(t)
            if not bt:
                continue
            gts = [(b[0], b[1]) for b in bt]
            hard = _matched_conf([d for d in dets.get(t, []) if d[4] >= pred_conf], gts) == 0.0
            sample_hard = hard and ref.split != "train"
            if not sample_hard and t % EASY_STEP:
                continue
            rng = np.random.default_rng([seed, ref.number, t])
            jobs = []        # (kind, x0, y0, clamped, points, meta)

            c = None
            if centre_mode == "predicted" and t in kf:
                c = (kf[t][0], kf[t][1], "kf", kf[t][2])
            if c is None:
                c = _centre(t, dets, balls, pred_conf, rng)
            if c is not None:
                x0, y0, clamped = clamp_window(c[0], c[1], W, H)
                pts, dropped = balls_in_window(bt, x0, y0)
                if not dropped:
                    kind = ("hard" if sample_hard else "easy") if pts else "outside"
                    conf = _matched_conf(dets.get(t, []), gts) if pts else _conf_in_window(dets.get(t, []), x0, y0)
                    rep = HARD_REPEAT if kind == "hard" and ref.split == "pool" else 1
                    if kind == "hard" and ref.split == "pool" and int(c[3]) >= 2:
                        rep *= gap2_repeat                # plan §4.3: more balls far from the last sighting
                    jobs.append((kind, x0, y0, clamped, pts,
                                 dict(centre_source=c[2], centre_gap=c[3], main_conf=conf, repeat=rep)))

            r = rng.random()
            neg_p = 1 / (1 - NEG_HARD_SHARE - NEG_RANDOM_SHARE) - 1     # negatives per positive
            if r < neg_p:
                cands = [(p[0], p[1]) for p in persons.get(t, [])] + lines.get(t, [])
                is_hard = rng.random() < NEG_HARD_SHARE / (NEG_HARD_SHARE + NEG_RANDOM_SHARE) and bool(cands)
                near = bt + balls.get(t - 1, []) + balls.get(t - 2, [])
                for _ in range(20):
                    if is_hard:
                        cx, cy = cands[rng.integers(len(cands))]
                        cx, cy = cx + rng.uniform(-64, 64), cy + rng.uniform(-64, 64)
                    else:
                        cx, cy = rng.uniform(0, W), rng.uniform(0, H)
                    x0, y0, clamped = clamp_window(cx, cy, W, H)
                    if not overlaps_any_ball(near, x0, y0):
                        jobs.append(("neg_hard" if is_hard else "neg_random", x0, y0, clamped, [],
                                     dict(centre_source="", centre_gap="", repeat=1,
                                          main_conf=_conf_in_window(dets.get(t, []), x0, y0))))
                        break

            if not jobs:
                continue
            prev = [f for f in (t - 2, t - 1) if f in files]
            idx = [t - 2 if t - 2 in files else t, t - 1 if t - 1 in files else t, t]
            imgs = [frame(f) for f in idx]
            for f in [k for k in frames_cache if k < t - 2]:
                del frames_cache[f]
            for kind, x0, y0, clamped, pts, meta in jobs:
                name = f"{t:06d}_{kind}.jpg"
                tile = np.concatenate([_cut(im, x0, y0) for im in imgs], axis=1)
                ok, enc = cv2.imencode(".jpg", tile, [cv2.IMWRITE_JPEG_QUALITY, quality])
                if not ok:
                    raise RuntimeError(f"jpeg encode failed: {ref.name} {name}")
                (seq_dir / name).write_bytes(enc.tobytes())
                p0 = pts[0] if pts else None
                rows.append(dict(split=ref.split, seq=ref.name, frame=t, kind=kind, hard=int(hard),
                                 x0=x0, y0=y0, clamped=int(clamped), n_prev=len(prev), n_balls=len(pts),
                                 ball_size=round(p0[2], 1) if p0 else "",
                                 ball_dx=round(p0[0] - CROP / 2, 1) if p0 else "",
                                 ball_dy=round(p0[1] - CROP / 2, 1) if p0 else "",
                                 balls=";".join(f"{x:.1f} {y:.1f} {s:.1f}" for x, y, s in pts),
                                 file=f"{ref.split}/{ref.name}/{name}",
                                 **{**meta, "main_conf": round(meta["main_conf"], 3)}))
        part.parent.mkdir(parents=True, exist_ok=True)
        tmp = part.with_suffix(".tmp")
        with open(tmp, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS)
            w.writeheader()
            w.writerows(rows)
        tmp.replace(part)
        return ref.name, rows, False
    finally:
        reader.close()


# --------------------------------------------------------------------------- stages

def stage_build(a):
    refs = [r for r in discover(a.src) if r.split in a.splits]
    if a.seqs:
        refs = [r for r in refs if r.name in a.seqs]
    missing = [r.name for r in refs if not (a.cache / r.split / f"{r.name}.json").exists()]
    if missing:
        raise SystemExit(f"no main-detector cache for {len(missing)} sequences, e.g. {missing[:3]}; run step 1")
    print(f"{len(refs)} sequences, {a.workers} workers", flush=True)
    a.out.mkdir(parents=True, exist_ok=True)
    tracker_cfg = None
    if a.tracker_config:
        tracker_cfg = json.loads(a.tracker_config.read_text()).get("tracker")
    if a.centre == "predicted":
        print(f"window centre: ball tracker prediction (motion {a.motion}, tracker config {a.tracker_config})", flush=True)
    jobs = [(r, str(a.cache), str(a.out), a.seed, a.pred_conf, a.jpeg_quality, a.centre,
             str(a.motion) if a.motion else None, a.gap2_repeat, tracker_cfg) for r in refs]
    t0, rows = time.time(), []
    with Pool(a.workers) as pool:
        for i, (name, r, cached) in enumerate(pool.imap_unordered(process_sequence, jobs), 1):
            rows.extend(r)
            print(f"[{i}/{len(refs)}] {name}: {len(r)} samples{' (cached)' if cached else ''}  "
                  f"({time.time() - t0:.0f}s)", flush=True)
    order = {s: i for i, s in enumerate(SPLITS)}
    rows.sort(key=lambda r: (order[r["split"]], r["seq"], int(r["frame"]), r["kind"]))
    with open(a.out / "manifest.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS)
        w.writeheader()
        w.writerows(rows)
    print(f"wrote {a.out / 'manifest.csv'} ({len(rows)} samples) in {(time.time() - t0) / 60:.1f} min\n")
    stage_stats(a)


def stage_stats(a):
    with open(a.out / "manifest.csv", newline="") as f:
        rows = list(csv.DictReader(f))
    kinds = ("hard", "easy", "outside", "neg_hard", "neg_random")
    lines = ["| split | samples | " + " | ".join(kinds) + " | hard frames sampled (any kind) | "
             "with repeat | GT-centred |", "|---" * (len(kinds) + 5) + "|"]
    for s in SPLITS:
        rs = [r for r in rows if r["split"] == s]
        if not rs:
            continue
        n = {k: sum(r["kind"] == k for r in rs) for k in kinds}
        hard_any = sum(r["hard"] == "1" and r["kind"] in ("hard", "outside") for r in rs)
        rep = sum(int(r["repeat"]) for r in rs)
        gt = sum(r["centre_source"] == "gt" for r in rs)
        lines.append(f"| {s} | {len(rs)} | " + " | ".join(str(n[k]) for k in kinds)
                     + f" | {hard_any} | {rep} | {gt} |")
    text = "\n".join(lines)
    print(text)
    (a.out / "stats.md").write_text(text + "\n")


def stage_preview(a):
    import cv2
    with open(a.out / "manifest.csv", newline="") as f:
        rows = [r for r in csv.DictReader(f) if r["split"] in a.splits]
    rng = np.random.default_rng(a.seed)
    kinds = sorted({r["kind"] for r in rows})
    pick = []
    for k in kinds:
        rs = [r for r in rows if r["kind"] == k]
        pick += [rs[i] for i in rng.choice(len(rs), min(a.n, len(rs)), replace=False)]
    outdir = a.out / "preview"
    outdir.mkdir(exist_ok=True)
    per_page = 8
    for p in range(0, len(pick), per_page):
        tiles = []
        for r in pick[p:p + per_page]:
            img = cv2.imread(str(a.out / r["file"]))
            for k in range(3):
                cv2.line(img, (CROP * k, 0), (CROP * k, CROP), (255, 255, 255), 2)
            for b in filter(None, r["balls"].split(";")):
                x, y, s = map(float, b.split())
                for k in range(3):     # same window position on every frame; the label is for t
                    col = (0, 255, 0) if k == 2 else (0, 160, 255)
                    cv2.circle(img, (int(x) + CROP * k, int(y)), int(max(s, 8)) + 6, col, 2)
            cv2.drawMarker(img, (CROP * 2 + CROP // 2, CROP // 2), (255, 0, 255), cv2.MARKER_CROSS, 20, 1)
            txt = (f"{r['split']} {r['seq']} f{r['frame']} {r['kind']} hard={r['hard']} "
                   f"src={r['centre_source']}{r['centre_gap']} conf={r['main_conf']} n_prev={r['n_prev']}")
            cv2.putText(img, txt, (8, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
            cv2.putText(img, txt, (8, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            tiles.append(cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA))
        path = outdir / f"page_{p // per_page:02d}.jpg"
        cv2.imwrite(str(path), np.concatenate(tiles, axis=0), [cv2.IMWRITE_JPEG_QUALITY, 90])
        print(path)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["build", "stats", "preview"])
    ap.add_argument("--src", type=Path, help="GSR source: sequence dirs or the split zips")
    ap.add_argument("--cache", type=Path, default=Path("data/outputs/ball_temporal/main_cache"))
    ap.add_argument("--out", type=Path, default=Path("data/outputs/ball_temporal/dataset"))
    ap.add_argument("--splits", nargs="+", default=list(SPLITS), choices=SPLITS)
    ap.add_argument("--seqs", nargs="*", default=[], help="only these sequences, e.g. SNGS-024 (smoke tests)")
    ap.add_argument("--workers", type=int, default=min(8, os.cpu_count() or 4))
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--pred_conf", type=float, default=0.15, help="main-detector confidence counted as a sighting")
    ap.add_argument("--jpeg_quality", type=int, default=95)
    ap.add_argument("--n", type=int, default=6, help="preview: samples per kind")
    ap.add_argument("--centre", choices=["last_seen", "predicted"], default="last_seen",
                    help="window centre: the last main sighting (as c5c trained) or the ball tracker's "
                         "prediction (docs/plans/BALL_IMPROVEMENT_PLAN.md §4.3); needs --motion")
    ap.add_argument("--motion", type=Path, default=None, help="camera_motion_cache.py output (--centre predicted)")
    ap.add_argument("--tracker_config", type=Path, default=None,
                    help="tracker_config.json from eval_ball_temporal.py track (its `tracker` section)")
    ap.add_argument("--gap2_repeat", type=int, default=1,
                    help="extra repeat factor for pool hard positives whose window centre is >= 2 frames old")
    a = ap.parse_args()
    if a.stage == "build" and not a.src:
        ap.error("--src is required")
    {"build": stage_build, "stats": stage_stats, "preview": stage_preview}[a.stage](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_temporal/train_ball_temporal.py
"""
train_ball_temporal.py -- step 3 of docs/path_b/plans/BALL_TEMPORAL_PLAN.md §9: train the temporal
heatmap specialist (heatmap_model.py) on the step 2 dataset (build_ball_temporal_dataset.py).

Single GPU. Standalone apart from heatmap_model.py (same folder). Needs torch, torchvision,
numpy, opencv-python, matplotlib.

  * `outside` crops (the runtime window misses the labelled ball) are not used for training and
    are not counted as empty crops: they are centred on a main-detector sighting, which is often a
    real but unlabelled ball (a spare ball by the boards). Labelled empty, they penalised correct
    detections (2026-10-14). Validation reports how often they fire, for reference.
  * Train on `train` + `pool`, sampled with replacement in proportion to the manifest's `repeat`
    (hard pool positives x3). One epoch = sum(repeat) samples.
  * Augmentation (plan §5): horizontal flip; the same brightness/contrast/colour gain on all three
    frames (on the GPU); a small translation (+-16 px, exposed border filled grey); and in 5% of samples
    t-1 and t-2 replaced by t (sequence start or a cut, plan §4). Optional (--motion_blur p, added
    2026-10-15, off by default so earlier runs reproduce): in a share p of samples, the same linear
    motion blur (3-9 px, random direction) on all three frames, like a fast pan or a fast ball.
  * Validation on `select` every epoch. Distance recall = peak within 15 px of a ball (plan §8),
    over positive crops. The selection score is its mean over the thresholds at which 1%, 2%, ...,
    10% of empty crops (`neg_hard`, `neg_random`) fire: a partial recall-vs-FPR area. Recall at a
    single 5%-FPR threshold swung 16-41% between epochs on noise in the empty crops' top scores, so
    it is reported but not selected on. Early stopping on the score, not on the loss.
  * AMP, AdamW, linear warm-up then cosine decay.

Output (<out>/): best.pt, last.pt (model + optimiser + epoch; resumable with --resume), history.csv,
curves.png, args.json, and DONE (finished: epochs or early stop) or INCOMPLETE (stopped by
--time_budget_h; run again with --resume).

Added for docs/plans/BALL_IMPROVEMENT_PLAN.md item 3 (2026-10-25):
  * --init <ckpt>: warm start from another checkpoint's weights (e.g. c5c/best.pt); optimiser and
    schedule start fresh. Use a lower --lr than the original run.
  * Multi-GPU: launch with `torchrun --nproc_per_node 2 train_ball_temporal.py ...`. Each GPU
    draws its own weighted sample stream (1/world of an epoch); validation, checkpoints and logs
    on rank 0 only.
  * --time_budget_h: stop after the epoch that would otherwise run past the budget (Kaggle's 12 h
    session limit), leaving last.pt to resume from.

    python scripts/ball_temporal/train_ball_temporal.py --data data/outputs/ball_temporal/dataset \
        --out runs/ball_temporal/c9 --channels 9
"""

from __future__ import annotations

import argparse
import csv
import json
import math
import os
import sys
import time
from datetime import timedelta
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

sys.path.insert(0, str(Path(__file__).resolve().parent))
from heatmap_model import (BallHeatmapNet, build_input, decode, focal_loss,  # noqa: E402
                           make_targets, offset_loss)

CROP = 640
MAX_BALLS = 4
MATCH_PX = 15.0
EMPTY_FPR = 0.05            # reported operating point: this share of empty crops fire
SCORE_FPRS = np.linspace(0.01, 0.10, 10)   # selection score: mean recall over these empty-crop FPRs
METRIC = "score_v3"         # checkpoints selected on an older metric are re-scored on --resume
SHIFT = 16
REPEAT_T = 0.05
KINDS = ("hard", "easy", "outside", "neg_hard", "neg_random")


def read_manifest(data: Path, splits) -> list[dict]:
    with open(data / "manifest.csv", newline="") as f:
        return [r for r in csv.DictReader(f) if r["split"] in splits]


def parse_balls(s: str) -> np.ndarray:
    pts = [list(map(float, b.split())) for b in s.split(";") if b]
    return np.array(pts, np.float32).reshape(-1, 3)


def motion_blur(frames: np.ndarray, rng) -> np.ndarray:
    """The same linear blur kernel (length 3-9 px, random angle) on each frame (3, H, W, 3)."""
    k = int(rng.integers(3, 10))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2] = 1.0
    rot = cv2.getRotationMatrix2D(((k - 1) / 2, (k - 1) / 2), float(rng.uniform(0, 180)), 1.0)
    kernel = cv2.warpAffine(kernel, rot, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return np.stack([cv2.filter2D(f, -1, kernel) for f in frames])


class TemporalCrops(Dataset):
    def __init__(self, data: Path, rows: list[dict], augment: bool, seed: int = 0, blur_p: float = 0.0):
        self.data, self.rows, self.augment, self.seed = data, rows, augment, seed
        self.blur_p = blur_p

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        img = cv2.imread(str(self.data / r["file"]), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(self.data / r["file"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        frames = np.stack([img[:, k * CROP:(k + 1) * CROP] for k in range(3)])   # (3, H, W, 3)
        pts = parse_balls(r["balls"])
        if self.augment:
            rng = np.random.default_rng()
            frames, pts = self._augment(frames, pts, rng)
            if self.blur_p and rng.random() < self.blur_p:
                frames = motion_blur(np.ascontiguousarray(frames), rng)
        out = np.full((MAX_BALLS, 3), -1, np.float32)
        out[:len(pts)] = pts[:MAX_BALLS]
        x = torch.from_numpy(np.ascontiguousarray(frames.transpose(0, 3, 1, 2).reshape(9, CROP, CROP)))
        return x, torch.from_numpy(out), KINDS.index(r["kind"]), i

    @staticmethod
    def _augment(frames, pts, rng):
        if rng.random() < REPEAT_T:
            frames = np.stack([frames[2]] * 3)
        if rng.random() < 0.5:
            frames = frames[:, :, ::-1]
            pts = pts.copy()
            pts[:, 0] = CROP - pts[:, 0]
        f = frames                                      # colour jitter runs on the GPU (color_jitter)
        if rng.random() < 0.5:
            dx, dy = (int(v) for v in rng.integers(-SHIFT, SHIFT + 1, 2))
            g = np.full_like(f, 114)
            sx, tx = (slice(0, CROP - dx), slice(dx, CROP)) if dx >= 0 else (slice(-dx, CROP), slice(0, CROP + dx))
            sy, ty = (slice(0, CROP - dy), slice(dy, CROP)) if dy >= 0 else (slice(-dy, CROP), slice(0, CROP + dy))
            g[:, ty, tx] = f[:, sy, sx]
            f = g
            pts = pts.copy()
            pts[:, 0] += dx
            pts[:, 1] += dy
            pts = pts[(pts[:, 0] >= 0) & (pts[:, 0] < CROP) & (pts[:, 1] >= 0) & (pts[:, 1] < CROP)]
        return f, pts


def color_jitter(x: torch.Tensor) -> torch.Tensor:
    """Per sample, the same contrast/brightness/colour gain on all three frames. uint8 (B, 9, H, W) in and out."""
    B, dev = x.shape[0], x.device
    a = torch.empty(B, 1, 1, 1, 1, device=dev).uniform_(0.75, 1.25)       # contrast
    b = torch.empty(B, 1, 1, 1, 1, device=dev).uniform_(-25, 25)          # brightness
    gain = torch.empty(B, 1, 3, 1, 1, device=dev).uniform_(0.92, 1.08)    # colour
    f = x.view(B, 3, 3, *x.shape[-2:]).float()
    f = ((f - 128) * a + 128 + b) * gain
    return f.clamp_(0, 255).round_().to(torch.uint8).view_as(x)


def step_loss(model, x, pts, channels):
    logits, off = model(build_input(x, channels))
    hm, ind, off_gt, mask = make_targets(pts, logits.shape[-1])
    lh = focal_loss(logits, hm)
    lo = offset_loss(off, ind, off_gt, mask)
    return lh + lo, lh, lo, logits, off


@torch.no_grad()
def validate(model, loader, channels, dev, amp):
    model.eval()
    confs, hits, pos, kinds, losses = [], [], [], [], []
    for x, pts, kind, _ in loader:
        x, pts = x.to(dev, non_blocking=True), pts.to(dev, non_blocking=True)
        with torch.autocast("cuda", enabled=amp):
            loss, *_rest, logits, off = step_loss(model, x, pts, channels)
        losses.append(loss.item())
        p = decode(logits, off)
        d = torch.cdist(p[:, None, :2], pts[..., :2]).squeeze(1)              # (B, K)
        d[pts[..., 2] <= 0] = float("inf")
        hits.append((d.min(1).values <= MATCH_PX).cpu())
        confs.append(p[:, 2].cpu())
        pos.append((pts[..., 2] > 0).any(1).cpu())
        kinds.append(kind)
    conf, hit, pos, kind = (torch.cat(v) for v in (confs, hits, pos, kinds))
    hard = kind == KINDS.index("hard")
    neg = kind >= KINDS.index("neg_hard")
    outside = conf[kind == KINDS.index("outside")]
    empty = conf[neg]
    thr = float(torch.quantile(empty, 1 - EMPTY_FPR)) if len(empty) else 0.5

    def rec(m, t):
        return float(((conf >= t) & hit & m).sum() / m.sum().clamp(min=1))

    def score(m):
        if not len(empty):
            return rec(m, 0.5)
        return float(np.mean([rec(m, float(torch.quantile(empty, 1 - f))) for f in SCORE_FPRS]))
    return {"val_loss": float(np.mean(losses)), "score": score(pos), "score_hard": score(pos & hard),
            "score_easy": score(pos & ~hard), "thr": thr,
            "recall": rec(pos, thr), "recall_hard": rec(pos & hard, thr),
            "recall_easy": rec(pos & ~hard, thr),
            "recall@0.5": rec(pos, 0.5),
            "outside_fire": float((outside >= thr).float().mean()) if len(outside) else 0.0, "empty_fpr@0.5": float((empty >= 0.5).float().mean()) if len(empty) else 0.0}


def plot(hist, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ep = [h["epoch"] for h in hist]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, [h["train_loss"] for h in hist], label="train")
    ax[0].plot(ep, [h["val_loss"] for h in hist], label="select")
    ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    for k in ("score", "score_hard", "score_easy", "recall", "recall@0.5"):
        pts = [(h["epoch"], float(h[k])) for h in hist if h.get(k) not in (None, "")]
        if pts:
            ax[1].plot(*zip(*pts), marker=".", label=k)
    ax[1].set_title(f"select ({MATCH_PX:.0f} px): score = mean recall at 1-10% empty-crop FPR")
    ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1); ax[1].legend()
    fig.tight_layout()
    fig.savefig(path, dpi=100)
    plt.close(fig)


def _init_distributed():
    """(world size, rank, local rank or None). torchrun sets WORLD_SIZE / RANK / LOCAL_RANK."""
    world = int(os.environ.get("WORLD_SIZE", "1"))
    if world <= 1:
        return 1, 0, None
    import torch.distributed as dist
    dist.init_process_group("nccl", timeout=timedelta(minutes=60))     # rank 0 validates while others wait
    local = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local)
    return world, dist.get_rank(), local


def _agree(flag: bool, dev, world: int) -> bool:
    """Rank 0's decision, on every rank."""
    if world <= 1:
        return flag
    import torch.distributed as dist
    t = torch.tensor([int(flag)], device=dev)
    dist.broadcast(t, 0)
    return bool(t.item())


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", type=Path, default=Path("data/outputs/ball_temporal/dataset"))
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--channels", type=int, choices=[5, 9], default=9)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--patience", type=int, default=8, help="epochs without a better select score")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--wd", type=float, default=1e-4)
    ap.add_argument("--warmup", type=float, default=1.0, help="warm-up length in epochs")
    ap.add_argument("--workers", type=int, default=12)
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--no_amp", action="store_true")
    ap.add_argument("--no_pretrained", action="store_true", help="skip the ImageNet weights download")
    ap.add_argument("--limit", type=int, default=0, help="smoke tests: samples per epoch and per validation")
    ap.add_argument("--resume", action="store_true", help="continue from <out>/last.pt")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--motion_blur", type=float, default=0.0, help="share of training samples given motion blur")
    ap.add_argument("--init", type=Path, default=None, help="warm start: load this checkpoint's weights (not optimiser)")
    ap.add_argument("--time_budget_h", type=float, default=0.0,
                    help="stop before an epoch would end past this many hours (0 = no limit); resume with --resume")
    a = ap.parse_args()

    world, rank, local = _init_distributed()
    main_rank = rank == 0
    log = (lambda *s, **k: print(*s, **k, flush=True)) if main_rank else (lambda *s, **k: None)
    t_start = time.time()
    torch.manual_seed(a.seed + rank)
    torch.backends.cudnn.benchmark = True
    dev = torch.device(f"cuda:{local}") if local is not None else torch.device(a.device)
    amp = not a.no_amp and dev.type == "cuda"
    a.out.mkdir(parents=True, exist_ok=True)
    if main_rank:
        for marker in ("DONE", "INCOMPLETE"):
            (a.out / marker).unlink(missing_ok=True)
        (a.out / "args.json").write_text(json.dumps({**{k: str(v) for k, v in vars(a).items()}, "world": world}, indent=1))

    tr_rows = [r for r in read_manifest(a.data, ("train", "pool")) if r["kind"] != "outside"]
    va_rows = read_manifest(a.data, ("select",))
    if a.limit:
        va_rows = [va_rows[i] for i in np.random.default_rng(a.seed).choice(len(va_rows), min(a.limit, len(va_rows)), replace=False)]
    weights = torch.tensor([float(r["repeat"]) for r in tr_rows])
    n_epoch = a.limit or int(weights.sum())
    sampler = WeightedRandomSampler(weights, max(n_epoch // world, a.batch), replacement=True,
                                    generator=torch.Generator().manual_seed(a.seed * 1000 + rank))
    tr = DataLoader(TemporalCrops(a.data, tr_rows, True, blur_p=a.motion_blur), a.batch, sampler=sampler, num_workers=a.workers,
                    pin_memory=True, drop_last=True, persistent_workers=a.workers > 0)
    va = DataLoader(TemporalCrops(a.data, va_rows, False), a.batch, shuffle=False, num_workers=a.workers,
                    pin_memory=True, persistent_workers=a.workers > 0) if main_rank else None
    log(f"train {len(tr_rows)} crops ({n_epoch} per epoch over {world} GPU(s)), select {len(va_rows)}, "
        f"{a.channels} channels, batch {a.batch} per GPU, amp {amp}")

    net = BallHeatmapNet(a.channels, pretrained=not (a.no_pretrained or a.init)).to(dev)   # channels_last was 4x slower (3090, torch 2.6)
    if a.init and not (a.resume and (a.out / "last.pt").exists()):
        ck0 = torch.load(a.init, map_location=dev, weights_only=False)
        if int(ck0.get("channels", a.channels)) != a.channels:
            raise SystemExit(f"--init {a.init} has {ck0.get('channels')} channels, this run {a.channels}")
        net.load_state_dict(ck0["model"])
        log(f"warm start from {a.init} (epoch {ck0.get('epoch')})")
    model = net
    opt = torch.optim.AdamW(net.parameters(), lr=a.lr, weight_decay=a.wd)
    steps_per_epoch = len(tr)
    total, warm = a.epochs * steps_per_epoch, int(a.warmup * steps_per_epoch)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / max(warm, 1) if s < warm else
        0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))) * 0.99 + 0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    hist, best, start, bad = [], -1.0, 1, 0
    if a.resume and (a.out / "last.pt").exists():
        ck = torch.load(a.out / "last.pt", map_location=dev, weights_only=False)
        net.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        hist, best, bad, start = ck["hist"], ck["best"], ck["bad"], ck["epoch"] + 1
        if ck.get("metric") != METRIC and main_rank:   # selected on the old single-threshold recall: re-score
            print("re-scoring best.pt and last.pt with the current selection score", flush=True)
            v_last = validate(model, va, a.channels, dev, amp)
            bk = torch.load(a.out / "best.pt", map_location=dev, weights_only=False)
            tmp = BallHeatmapNet(a.channels, pretrained=False).to(dev)
            tmp.load_state_dict(bk["model"])
            v_best = validate(tmp, va, a.channels, dev, amp)
            del tmp
            for e, v in ((bk["epoch"], v_best), (ck["epoch"], v_last)):
                for h in hist:
                    if h["epoch"] == e:
                        h.update({k: v[k] for k in ("score", "score_hard", "score_easy")})
            print(f"  best.pt (epoch {bk['epoch']}) score {v_best['score']:.2%}, "
                  f"last.pt (epoch {ck['epoch']}) score {v_last['score']:.2%}", flush=True)
            if v_last["score"] > v_best["score"]:
                best, bad = v_last["score"], 0
                torch.save({"model": model.state_dict(), "channels": a.channels, "epoch": ck["epoch"],
                            "metrics": {**hist[-1], **v_last}}, a.out / "best.pt")
            else:
                best, bad = v_best["score"], ck["epoch"] - bk["epoch"]
        log(f"resumed at epoch {start}, best select score {best:.4f}, {bad} epochs since")

    if world > 1:
        model = torch.nn.parallel.DistributedDataParallel(net, device_ids=[local])
    epoch_min = []
    stopped_by_budget = False
    for ep in range(start, a.epochs + 1):
        model.train()
        t0, tl, n = time.time(), 0.0, 0
        for i, (x, pts, _, _) in enumerate(tr, 1):
            x = color_jitter(x.to(dev, non_blocking=True))
            pts = pts.to(dev, non_blocking=True)
            with torch.autocast("cuda", enabled=amp):
                loss, lh, lo, *_ = step_loss(model, x, pts, a.channels)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            tl, n = tl + loss.item(), n + 1
            if i % 100 == 0 or i == steps_per_epoch:
                per_it = (time.time() - t0) / i
                eta_ep = per_it * (steps_per_epoch - i) / 60
                mem = f" mem {torch.cuda.max_memory_allocated(dev) / 2**30:.1f}G" if dev.type == "cuda" else ""
                log(f"  ep {ep}/{a.epochs} {i}/{steps_per_epoch} loss {tl / n:.4f} (hm {lh.item():.3f} off {lo.item():.3f}) "
                    f"lr {sched.get_last_lr()[0]:.2e} {per_it:.2f}s/it, epoch ETA {eta_ep:.1f} min{mem}")
        stop = False
        if main_rank:
            v = validate(net, va, a.channels, dev, amp)
            h = {"epoch": ep, "train_loss": tl / max(n, 1), **v, "minutes": (time.time() - t0) / 60}
            hist.append(h)
            epoch_min.append(h["minutes"])
            improved = v["score"] > best
            if improved:
                best, bad = v["score"], 0
                torch.save({"model": net.state_dict(), "channels": a.channels, "epoch": ep, "metrics": h},
                           a.out / "best.pt")
            else:
                bad += 1
            torch.save({"model": net.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "channels": a.channels, "epoch": ep, "hist": hist,
                        "best": best, "bad": bad, "metric": METRIC}, a.out / "last.pt")
            with open(a.out / "history.csv", "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=list(h), restval="")
                w.writeheader()
                w.writerows(hist)
            plot(hist, a.out / "curves.png")
            left = (a.epochs - ep) * float(np.mean(epoch_min[-3:]))
            log(f"epoch {ep}/{a.epochs}: train {h['train_loss']:.4f} select loss {v['val_loss']:.4f} | score {v['score']:.2%} "
                f"(hard {v['score_hard']:.2%}, easy {v['score_easy']:.2%}) | recall {v['recall']:.2%} "
                f"(hard {v['recall_hard']:.2%}, easy {v['recall_easy']:.2%}) at thr {v['thr']:.3f} | "
                f"outside fire {v['outside_fire']:.0%} | recall@0.5 {v['recall@0.5']:.2%}, empty FPR@0.5 {v['empty_fpr@0.5']:.2%} | "
                f"{h['minutes']:.1f} min{'  *best*' if improved else ''} | run ETA {left / 60:.1f} h")
            if bad >= a.patience:
                log(f"early stop: no better select score for {a.patience} epochs")
                stop = True
            elif a.time_budget_h and ep < a.epochs:
                used_h = (time.time() - t_start) / 3600
                next_h = 1.15 * max(epoch_min[-3:]) / 60
                if used_h + next_h > a.time_budget_h:
                    log(f"time budget: {used_h:.2f} h used, next epoch ~{next_h:.2f} h, budget {a.time_budget_h} h -> "
                        f"stopping after epoch {ep}; continue with --resume")
                    stop = stopped_by_budget = True
        stop = _agree(stop, dev, world)
        if stop:
            break
    if main_rank:
        (a.out / ("INCOMPLETE" if stopped_by_budget else "DONE")).write_text(
            json.dumps({"best_score": best, "epochs_run": len(hist), "hours": (time.time() - t_start) / 3600}))
        log(f"best select score {best:.2%}; {a.out / 'best.pt'} ({'INCOMPLETE' if stopped_by_budget else 'DONE'})")
    if world > 1:
        import torch.distributed as dist
        dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_temporal/export_ball_specialist.py
"""
export_ball_specialist.py -- step 5 of docs/path_b/plans/BALL_TEMPORAL_PLAN.md §9: export a
temporal ball specialist checkpoint to ONNX with a sidecar, and check the ONNX against PyTorch.

The TensorRT engine is not built here. The pipeline builds it from this ONNX on first launch on
each host (gsr_pipeline/runtime/trt_runner.ensure_engine), as <stem>_fp16.engine beside it.

Graph (static batch 1):
  input  "frames"  float32 (1, 9, 640, 640): crops of frames t-2, t-1, t, RGB, 0..255, in that
                   order. Normalisation and the 5-channel difference images are inside the graph,
                   so the runtime only cuts and stacks crops.
  output "heatmap" float32 (1, 1, 320, 320): logits; sigmoid = ball-centre confidence per cell.
  output "offset"  float32 (1, 2, 320, 320): sub-pixel offset per cell.
  Decode (host): peak cell (ix, iy), point = ((ix, iy) + offset[:, iy, ix]) * 2 in crop pixels,
  confidence = sigmoid(peak).

Sidecar <onnx>.json: checkpoint, epoch, channels, input/output contract, stride, the system
threshold from step 4, and the ONNX sha256 (the engine sidecar is checked against it).

Parity: ONNX Runtime (CPU, and GPU if available) against PyTorch FP32 on real crops from the
step 2 dataset. Pass: every decoded point within 0.5 px and confidence within 1e-3.

    python scripts/ball_temporal/export_ball_specialist.py --ckpt runs/ball_temporal/c5b/best.pt \
        --out runs/ball_temporal/c5b/ball_specialist_c5b.onnx --threshold 0.75
"""

from __future__ import annotations

import argparse
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

sys.path.insert(0, str(Path(__file__).resolve().parent))
from heatmap_model import STRIDE, BallHeatmapNet, build_input, decode  # noqa: E402

CROP = 640
POINT_TOL_PX = 0.5
CONF_TOL = 1e-3


class ExportWrapper(nn.Module):
    """float 0..255 frames in, (heatmap logits, offset) out; preprocessing inside the graph."""

    def __init__(self, net: BallHeatmapNet):
        super().__init__()
        self.net = net

    def forward(self, frames):
        return self.net(build_input(frames, self.net.channels))


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while b := f.read(1 << 20):
            h.update(b)
    return h.hexdigest()


def load_crops(data: Path, n: int, seed: int) -> list[np.ndarray]:
    """n crops (9, 640, 640) float32 from the step 2 test split, mixed kinds."""
    import csv

    import cv2
    with open(data / "manifest.csv", newline="") as f:
        rows = [r for r in csv.DictReader(f) if r["split"] == "test"]
    rng = np.random.default_rng(seed)
    out = []
    for i in rng.choice(len(rows), min(n, len(rows)), replace=False):
        img = cv2.cvtColor(cv2.imread(str(data / rows[i]["file"])), cv2.COLOR_BGR2RGB)
        f = np.stack([img[:, k * CROP:(k + 1) * CROP] for k in range(3)])     # (3, H, W, 3)
        out.append(f.transpose(0, 3, 1, 2).reshape(9, CROP, CROP).astype(np.float32))
    return out


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--ckpt", type=Path, required=True)
    ap.add_argument("--out", type=Path, required=True, help="ONNX path; the sidecar is written beside it")
    ap.add_argument("--threshold", type=float, required=True, help="system threshold from step 4, recorded in the sidecar")
    ap.add_argument("--data", type=Path, default=Path("data/outputs/ball_temporal/dataset"))
    ap.add_argument("--n", type=int, default=64, help="parity crops")
    ap.add_argument("--opset", type=int, default=17)
    ap.add_argument("--batch", type=int, default=1,
                    help="static batch size (3 = the 3-window variant, docs/plans/BALL_IMPROVEMENT_PLAN.md §4.1). "
                         "gsr_pipeline/ball_specialist.py loads batch 1 only until that variant is wired.")
    ap.add_argument("--report", type=Path, default=None, help="also write the parity results here as JSON")
    a = ap.parse_args()

    ck = torch.load(a.ckpt, map_location="cpu", weights_only=False)
    net = BallHeatmapNet(ck["channels"], pretrained=False)
    net.load_state_dict(ck["model"])
    model = ExportWrapper(net).eval()

    a.out.parent.mkdir(parents=True, exist_ok=True)
    B = a.batch
    dummy = torch.zeros(B, 9, CROP, CROP)
    torch.onnx.export(model, dummy, str(a.out), input_names=["frames"], output_names=["heatmap", "offset"],
                      opset_version=a.opset, do_constant_folding=True, dynamo=False)
    meta = {
        "schema": 1,
        "model": "ball_temporal_heatmap",
        "checkpoint": str(a.ckpt).replace("\\", "/"),
        "epoch": ck.get("epoch"),
        "channels": ck["channels"],
        "input": {"name": "frames", "shape": [B, 9, CROP, CROP], "dtype": "float32",
                  "layout": "RGB crops of frames t-2, t-1, t, values 0..255; preprocessing in graph"},
        "outputs": {"heatmap": {"shape": [B, 1, CROP // STRIDE, CROP // STRIDE], "activation": "sigmoid (host)"},
                    "offset": {"shape": [B, 2, CROP // STRIDE, CROP // STRIDE]}},
        "stride": STRIDE,
        "crop": CROP,
        "threshold": a.threshold,
        "onnx_sha256": sha256(a.out),
    }
    Path(str(a.out) + ".json").write_text(json.dumps(meta, indent=2) + "\n")
    print(f"wrote {a.out} ({a.out.stat().st_size / 1e6:.1f} MB) and its sidecar")

    import onnxruntime as ort
    crops = load_crops(a.data, a.n, seed=0)
    with torch.no_grad():
        ref = [decode(*model(torch.from_numpy(c)[None]))[0].numpy() for c in crops]
    providers = [p for p in ("CPUExecutionProvider", "CUDAExecutionProvider") if p in ort.get_available_providers()]
    ok_all, results = True, {}
    n_full = len(crops) - len(crops) % B          # the graph is static: whole batches only
    for prov in providers:
        sess = ort.InferenceSession(str(a.out), providers=[prov])
        if prov not in sess.get_providers():
            print(f"{prov}: requested but not active, skipped")
            continue
        dp, dc = [], []
        for k in range(0, n_full, B):
            hm, off = sess.run(None, {"frames": np.stack(crops[k:k + B])})
            ps = decode(torch.from_numpy(hm), torch.from_numpy(off)).numpy()
            for p, r in zip(ps, ref[k:k + B]):
                dp.append(float(np.hypot(*(p[:2] - r[:2]))))
                dc.append(float(abs(p[2] - r[2])))
        ok = max(dp) <= POINT_TOL_PX and max(dc) <= CONF_TOL
        ok_all &= ok
        results[prov] = {"crops": len(dp), "max_point_diff_px": max(dp), "max_conf_diff": max(dc), "pass": ok}
        print(f"{prov}: {len(dp)} crops (batch {B}), max point diff {max(dp):.4f} px, max conf diff {max(dc):.2e} "
              f"-> {'PASS' if ok else 'FAIL'}")
    if a.report:
        a.report.parent.mkdir(parents=True, exist_ok=True)
        a.report.write_text(json.dumps({"onnx": str(a.out), "batch": B, "threshold": a.threshold,
                                        "providers": results, "pass": ok_all}, indent=1))
    sys.exit(0 if ok_all else 1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/eval/ball_det/eval_ball_temporal.py
"""
eval_ball_temporal.py -- step 4 of docs/path_b/plans/BALL_TEMPORAL_PLAN.md §9: gates 1-3 (plan §8)
for a temporal ball specialist checkpoint (scripts/ball_temporal/train_ball_temporal.py).

Standalone apart from scripts/ball_temporal/ (gsr_io, heatmap_model, build_ball_temporal_dataset,
train_ball_temporal). Needs torch, numpy, opencv-python. Reads the main-detector cache (step 1) and
the crop dataset (step 2).

  crops   gates 1-2: the step 2 crops of a split. Specialist vs main detector on the same crops,
          all positives and hard positives, plus false positives on negative crops.
  system  gate 3: main + specialist on every frame of a split, simulated as the runtime runs it.
  all     both, threshold chosen on `select`, applied unchanged to `test`; writes report.md.

System simulation, per sequence, frame by frame (implementation plan §5.1, temporal plan §6):
  * the main detector's top ball at conf >= --main_conf is the output, and its position is the
    last sighting;
  * otherwise, if the last sighting is at most --max_miss frames old, the specialist runs on
    frames t-2, t-1, t cut by a 640x640 window centred on it (shift-clamped, no camera shift);
    its peak above the threshold is the output;
  * --chain fused (the runtime): a specialist detection becomes the new sighting (window
    re-centres, gap resets). --chain main (as the training crops were built): only main-detector
    sightings place the window.
Scoring: 15 px centre match. recall = hit frames / ball-labelled frames. precision = hits / outputs
on ball-labelled frames (outputs on frames without a ball label are counted separately: those
frames may show an unlabelled ball). Longest miss = longest run of ball-labelled frames not hit.
Threshold (--rule, chosen on `select`, frozen for `test`):
  spec_precision (default, 2026-10-15): the lowest threshold at which >= --spec_precision of the
    specialist's own outputs on ball-labelled frames are on the labelled ball. Independent of the
    main detector's precision, which differs between splits (88.3% select, 85.3% test).
  main_precision (the first rule): the highest recall whose system precision is >= the main
    detector's own on `select`.

    python scripts/eval/ball_det/eval_ball_temporal.py all --ckpt runs/ball_temporal/c5b/best.pt \
        --src data/datasets/SoccerNetGS --out data/outputs/ball_temporal/eval/c5b
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch

ROOT = Path(__file__).resolve().parents[3]
sys.path.insert(0, str(ROOT / "scripts" / "ball_temporal"))
sys.path.insert(0, str(Path(__file__).resolve().parent))
from build_ball_temporal_dataset import CROP, clamp_window  # noqa: E402
from gsr_io import SeqReader, decode, discover, parse_labels  # noqa: E402
from heatmap_model import BallHeatmapNet, build_input, decode as decode_hm  # noqa: E402

MATCH_PX = 15.0
FPS = 25
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.05), 2)
GAP_BUCKETS = ((1, 1), (2, 2), (3, 3), (4, 5), (6, 10), (11, 15), (16, 25))


def load_model(ckpt: Path, dev):
    ck = torch.load(ckpt, map_location=dev, weights_only=False)
    m = BallHeatmapNet(ck["channels"], pretrained=False).to(dev).eval()
    m.load_state_dict(ck["model"])
    return m, ck["channels"], ck.get("epoch")


def _near(p, gts) -> bool:
    return any(np.hypot(p[0] - g[0], p[1] - g[1]) <= MATCH_PX for g in gts)


# --------------------------------------------------------------------------- gates 1-2: crops

@torch.no_grad()
def crop_scores(model, channels, data: Path, split: str, dev, workers: int):
    """Specialist peak per crop of a split -> rows with conf, hit (and the manifest's main_conf)."""
    from torch.utils.data import DataLoader
    from train_ball_temporal import TemporalCrops, read_manifest
    rows = read_manifest(data, (split,))
    dl = DataLoader(TemporalCrops(data, rows, False), 32, num_workers=workers)
    for x, pts, _, idx in dl:
        x, pts = x.to(dev), pts.to(dev)
        with torch.autocast("cuda", enabled=dev.type == "cuda"):
            logits, off = model(build_input(x, channels))
        p = decode_hm(logits, off)
        d = torch.cdist(p[:, None, :2], pts[..., :2]).squeeze(1)
        d[pts[..., 2] <= 0] = float("inf")
        hit = (d.min(1).values <= MATCH_PX).cpu().numpy()
        for j, i in enumerate(idx.tolist()):
            rows[i]["spec_conf"] = float(p[j, 2])
            rows[i]["spec_hit"] = bool(hit[j])
    return rows


def crop_report(rows, spec_thr: float, main_conf: float) -> dict:
    """Gates 1-2. Main detector on a crop: a main ball >= main_conf within 15 px of the GT ball
    (positives), or any main ball >= main_conf inside the window (negatives); from the manifest."""
    out = {}
    groups = {"all positives": lambda r: r["kind"] in ("hard", "easy"),
              "hard": lambda r: r["kind"] == "hard", "easy": lambda r: r["kind"] == "easy"}
    for name, f in groups.items():
        rs = [r for r in rows if f(r)]
        out[name] = {"n": len(rs),
                     "main": float(np.mean([float(r["main_conf"]) >= main_conf for r in rs])) if rs else 0.0,
                     "spec": float(np.mean([r["spec_hit"] and r["spec_conf"] >= spec_thr for r in rs])) if rs else 0.0}
    neg = [r for r in rows if r["kind"].startswith("neg")]
    out["negatives fire"] = {"n": len(neg),
                             "main": float(np.mean([float(r["main_conf"]) >= main_conf for r in neg])),
                             "spec": float(np.mean([r["spec_conf"] >= spec_thr for r in neg]))}
    outside = [r for r in rows if r["kind"] == "outside"]
    out["outside fire"] = {"n": len(outside),
                           "main": float(np.mean([float(r["main_conf"]) >= main_conf for r in outside])) if outside else 0.0,
                           "spec": float(np.mean([r["spec_conf"] >= spec_thr for r in outside])) if outside else 0.0}
    return out


def neg_fpr_threshold(rows, fpr: float) -> float:
    neg = np.array([r["spec_conf"] for r in rows if r["kind"].startswith("neg")])
    return float(np.quantile(neg, 1 - fpr))


# --------------------------------------------------------------------------- gate 3: system

class Specialist:
    """Runs the model on one window of frames t-2, t-1, t; memoised by (frame, x0, y0)."""

    def __init__(self, model, channels, dev):
        self.model, self.channels, self.dev = model, channels, dev
        self.memo: dict = {}
        self.calls = 0
        self.ms = 0.0

    @torch.no_grad()
    def __call__(self, key, frames, x0, y0):
        if key in self.memo:
            return self.memo[key]
        x = np.concatenate([f[y0:y0 + CROP, x0:x0 + CROP, ::-1] for f in frames], axis=2)   # BGR->RGB, HxWx9
        x = torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))[None].to(self.dev)
        t0 = time.perf_counter()
        with torch.autocast("cuda", enabled=self.dev.type == "cuda"):
            logits, off = self.model(build_input(x, self.channels))
        p = decode_hm(logits, off)[0].tolist()
        if self.dev.type == "cuda":
            torch.cuda.synchronize()
        self.ms += (time.perf_counter() - t0) * 1000
        self.calls += 1
        self.memo[key] = (p[0] + x0, p[1] + y0, p[2])
        return self.memo[key]


class SeqData:
    """One sequence: labels, cached main detections, and a small decoded-frame cache."""

    def __init__(self, ref, cache_dir: Path):
        self.ref = ref
        self.reader = SeqReader(ref)
        self.im_dir, self.files, balls, *_rest, (self.W, self.H) = parse_labels(self.reader.labels())
        self.gt = {f: [(b[0], b[1]) for b in bs] for f, bs in balls.items()}
        cache = json.loads((cache_dir / ref.split / f"{ref.name}.json").read_text())["frames"]
        self.dets = {int(k): v["balls"] for k, v in cache.items()}
        self._frames: dict = {}

    def frame(self, f):
        if f not in self._frames:
            if len(self._frames) > 8:
                self._frames.pop(min(self._frames))
            self._frames[f] = decode(self.reader.read(f"{self.im_dir}/{self.files[f]}"))
        return self._frames[f]

    def window_frames(self, t):
        return [self.frame(f if f in self.files else t) for f in (t - 2, t - 1, t)]


def simulate(sd: SeqData, spec: Specialist | None, thr: float, a) -> list[dict]:
    """-> per frame {t, x, y, src, gap} (x None when no output)."""
    out, last = [], None                        # last sighting: (x, y, frame)
    for t in sorted(sd.files):
        m = [b for b in sd.dets.get(t, []) if b[4] >= a.main_conf]
        if m:
            last = (m[0][0], m[0][1], t)
            out.append(dict(t=t, x=m[0][0], y=m[0][1], src="main", gap=None))
            continue
        rec = dict(t=t, x=None, y=None, src="", gap=None)
        if spec is not None and last is not None and t - last[2] <= a.max_miss:
            x0, y0, _ = clamp_window(last[0], last[1], sd.W, sd.H)
            rec["gap"] = t - last[2]
            key = (sd.ref.name, t, x0, y0)
            frames = None if key in spec.memo else sd.window_frames(t)
            px, py, conf = spec(key, frames, x0, y0)
            if conf >= thr:
                rec.update(x=px, y=py, src="spec")
                if a.chain == "fused":
                    last = (px, py, t)
        out.append(rec)
    return out


def score(sd_list, outs) -> dict:
    n_gt = hit = pred_gt = pred_nogt = spec_out = spec_hit = 0
    misses, gap_n, gap_hit = [], defaultdict(int), defaultdict(int)
    for sd, out in zip(sd_list, outs):
        run, longest = 0, 0
        for r in out:
            gts = sd.gt.get(r["t"])
            ok = bool(gts) and r["x"] is not None and _near((r["x"], r["y"]), gts)
            if gts:
                n_gt += 1
                hit += ok
                pred_gt += r["x"] is not None
                if r["src"] == "spec":
                    spec_out += 1
                    spec_hit += ok
                if r["gap"] is not None:
                    b = next((f"{lo}-{hi}" if lo != hi else str(lo) for lo, hi in GAP_BUCKETS if lo <= r["gap"] <= hi), ">25")
                    gap_n[b] += 1
                    gap_hit[b] += ok
                run = 0 if ok else run + 1
                longest = max(longest, run)
            elif r["x"] is not None:
                pred_nogt += 1
        misses.append(longest / FPS)
    return {"gt_frames": n_gt, "recall": hit / max(n_gt, 1), "precision": hit / max(pred_gt, 1),
            "outputs_unlabelled": pred_nogt, "spec_outputs": spec_out,
            "spec_precision": spec_hit / max(spec_out, 1), "spec_added": spec_hit / max(n_gt, 1),
            "longest_miss_mean_s": float(np.mean(misses)), "longest_miss_max_s": float(np.max(misses)),
            "by_gap": {k: (gap_hit[k] / gap_n[k], gap_n[k]) for k in sorted(gap_n, key=lambda s: int(s.split("-")[0].lstrip(">")))}}


def run_system(split, spec, a, thresholds):
    refs = [r for r in discover(a.src) if r.split == split]
    sds = [SeqData(r, a.cache) for r in refs]
    base = score(sds, [simulate(sd, None, 1.0, a) for sd in sds])
    res = {}
    for thr in thresholds:
        t0 = time.time()
        res[float(thr)] = score(sds, [simulate(sd, spec, float(thr), a) for sd in sds])
        print(f"  {split} thr {thr:.2f}: recall {res[float(thr)]['recall']:.2%} precision "
              f"{res[float(thr)]['precision']:.2%} ({time.time() - t0:.0f}s, {spec.calls} specialist calls)", flush=True)
    for sd in sds:
        sd.reader.close()
    return base, res


# --------------------------------------------------------------------------- report

def pct(v):
    return f"{v:.2%}"


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["all", "track"],
                    help="all: gates 1-3 (this file). track: ball tracker + window variants "
                         "(ball_system.py, docs/plans/BALL_IMPROVEMENT_PLAN.md items 2-3)")
    ap.add_argument("--ckpt", type=Path, required=True)
    ap.add_argument("--src", type=Path, default=Path("data/datasets/SoccerNetGS"))
    ap.add_argument("--cache", type=Path, default=Path("data/outputs/ball_temporal/main_cache"))
    ap.add_argument("--data", type=Path, default=Path("data/outputs/ball_temporal/dataset"))
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--main_conf", type=float, default=0.15, help="main-detector output threshold")
    ap.add_argument("--max_miss", type=int, default=15, help="frames since the last sighting the specialist still runs")
    ap.add_argument("--chain", choices=["fused", "main"], default="fused")
    ap.add_argument("--rule", choices=["spec_precision", "main_precision"], default="spec_precision")
    ap.add_argument("--spec_precision", type=float, default=0.90,
                    help="spec_precision rule: required share of specialist outputs on the labelled ball")
    ap.add_argument("--crop_fpr", type=float, default=0.05, help="gates 1-2 threshold: this share of select negatives fire")
    ap.add_argument("--test_sweep", action="store_true",
                    help="also report the test curve at every threshold: a diagnostic, never for choosing")
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--motion", type=Path, default=None, help="track: camera_motion_cache.py output (none = no motion)")
    ap.add_argument("--system_thr", type=float, default=0.70,
                    help="track: the as-wired specialist threshold for the baseline (c5c's sidecar: 0.70)")
    ap.add_argument("--search_passes", type=int, default=2, help="track: coordinate-search passes over the tracker knobs")
    ap.add_argument("--max_seqs", type=int, default=0, help="track: first N sequences per split (smoke tests)")
    a = ap.parse_args()
    dev = torch.device(a.device)
    torch.backends.cudnn.benchmark = True
    a.out.mkdir(parents=True, exist_ok=True)
    model, channels, epoch = load_model(a.ckpt, dev)
    print(f"{a.ckpt} (epoch {epoch}, {channels} channels), chain {a.chain}, max_miss {a.max_miss}", flush=True)
    if a.stage == "track":
        from ball_system import stage_track
        a.chain = "fused"
        stage_track(a, model, channels, dev)
        return

    # gates 1-2
    sel_rows = crop_scores(model, channels, a.data, "select", dev, a.workers)
    crop_thr = neg_fpr_threshold(sel_rows, a.crop_fpr)
    test_rows = crop_scores(model, channels, a.data, "test", dev, a.workers)
    crops = crop_report(test_rows, crop_thr, a.main_conf)
    print(f"crop threshold {crop_thr:.3f} ({a.crop_fpr:.0%} of select negatives fire)", flush=True)

    # gate 3
    spec = Specialist(model, channels, dev)
    sel_base, sel = run_system("select", spec, a, THRESHOLDS)
    if a.rule == "spec_precision":
        ok = [t for t, r in sel.items() if r["spec_precision"] >= a.spec_precision]
        thr = min(ok) if ok else max(sel)
        rule_text = (f"the lowest `select` threshold at which >= {a.spec_precision:.0%} of the specialist's "
                     f"outputs are on the labelled ball")
    else:
        ok = [t for t, r in sel.items() if r["precision"] >= sel_base["precision"]]
        thr = max(ok, key=lambda t: sel[t]["recall"]) if ok else max(sel, key=lambda t: sel[t]["precision"])
        rule_text = f"the highest `select` recall with precision >= the main detector's on `select` ({sel_base['precision']:.2%})"
    print(f"system threshold {thr:.2f} (select: recall {sel[thr]['recall']:.2%}, precision "
          f"{sel[thr]['precision']:.2%}; main alone {sel_base['recall']:.2%} / {sel_base['precision']:.2%})", flush=True)
    spec.calls, spec.ms = 0, 0.0
    test_base, test = run_system("test", spec, a, THRESHOLDS if a.test_sweep else [thr])
    t = test[thr]
    ms = spec.ms / max(spec.calls, 1)      # with --test_sweep, averaged over every pass

    g1 = crops["all positives"]["spec"] > crops["all positives"]["main"]
    g3 = t["recall"] >= test_base["recall"] + 0.10 and t["precision"] >= test_base["precision"]
    L = [f"# Temporal ball specialist: test gates", "",
         f"Checkpoint `{a.ckpt}` (epoch {epoch}, {channels} channels). Main detector cache "
         f"`{a.cache}` (imgsz 1280, output conf >= {a.main_conf}). 15 px centre match. Thresholds "
         f"chosen on `select`, frozen for `test` (15 sequences).", "",
         "## Gates 1-2: crops (step 2 test crops)", "",
         f"Specialist threshold {crop_thr:.3f}: {a.crop_fpr:.0%} of `select` negative crops fire. "
         f"Main detector on a crop: a main ball >= {a.main_conf} within 15 px (positives) or inside the window (negatives).", "",
         "| crops | n | main detector | specialist |", "|---|---|---|---|"]
    for k, v in crops.items():
        L.append(f"| {k} | {v['n']} | {pct(v['main'])} | {pct(v['spec'])} |")
    L += ["", f"Gate 1 (specialist beats main on all test crops): **{'PASS' if g1 else 'FAIL'}**. "
              f"Gate 2 (hard crops): reported above.", "",
          f"## Gate 3: system, every test frame (chain `{a.chain}`, max_miss {a.max_miss})", "",
          f"System threshold {thr:.2f} (rule `{a.rule}`): {rule_text}.", "",
          "| | recall | precision | outputs on unlabelled frames | longest miss mean / max |", "|---|---|---|---|---|",
          f"| main detector | {pct(test_base['recall'])} | {pct(test_base['precision'])} | {test_base['outputs_unlabelled']} | "
          f"{test_base['longest_miss_mean_s']:.1f} s / {test_base['longest_miss_max_s']:.1f} s |",
          f"| main + specialist | {pct(t['recall'])} | {pct(t['precision'])} | {t['outputs_unlabelled']} | "
          f"{t['longest_miss_mean_s']:.1f} s / {t['longest_miss_max_s']:.1f} s |", "",
          f"Recall {(t['recall'] - test_base['recall']) * 100:+.1f} points, precision "
          f"{(t['precision'] - test_base['precision']) * 100:+.1f} points. Specialist outputs on labelled frames: "
          f"{t['spec_outputs']}, {pct(t['spec_precision'])} of them on the labelled ball.", "",
          f"Gate 3 (recall >= main + 10 points, precision >= main): **{'PASS' if g3 else 'FAIL'}**.", "",
          "Specialist recall on labelled frames the main detector missed, by frames since the last sighting:", "",
          "| gap | frames | recall |", "|---|---|---|"]
    for k, (r, n) in t["by_gap"].items():
        L.append(f"| {k} | {n} | {pct(r)} |")
    L += ["", f"`select` sweep (main alone: recall {pct(sel_base['recall'])}, precision {pct(sel_base['precision'])}):", "",
          "| threshold | recall | precision | specialist outputs on the ball |", "|---|---|---|---|"]
    for k, r in sel.items():
        L.append(f"| {k:.2f} | {pct(r['recall'])} | {pct(r['precision'])} | {pct(r['spec_precision'])} |")
    if a.test_sweep:
        L += ["", "`test` sweep, **diagnostic only** (the gate uses the `select` threshold above):", "",
              "| threshold | recall | precision |", "|---|---|---|"]
        for k, r in test.items():
            L.append(f"| {k:.2f} | {pct(r['recall'])} | {pct(r['precision'])} |")
    L += ["", f"Specialist PyTorch FP16 forward + decode, batch 1, this machine: {ms:.1f} ms per call "
              f"({spec.calls} calls on test). Not the gate 4 figure (that is TensorRT, step 5)."]
    text = "\n".join(L)
    print("\n" + text)
    (a.out / "report.md").write_text(text + "\n", encoding="utf-8")
    (a.out / "results.json").write_text(json.dumps(
        {"ckpt": str(a.ckpt), "epoch": epoch, "channels": channels, "chain": a.chain, "max_miss": a.max_miss,
         "rule": a.rule, "spec_precision_target": a.spec_precision,
         "crop_thr": crop_thr, "crops": crops, "system_thr": thr, "select_main": sel_base, "select": sel,
         "test_main": test_base, "test": t, "ms_per_call": ms, "gate1": g1, "gate3": g3}, indent=1, default=str))
    with open(a.out / "test_crops.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(test_rows[0]))
        w.writeheader()
        w.writerows(test_rows)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/eval/ball_det/ball_system.py
"""
ball_system.py -- main detector + c5c + ball tracker, simulated frame by frame, for
docs/plans/BALL_IMPROVEMENT_PLAN.md items 2-3 (the `track` stage of eval_ball_temporal.py, and
diagnose_ball_misses.py). Builds on eval_ball_temporal.py's Specialist (memoised windows) and
SeqData (labels, cached main detections, decoded-frame cache).

Per frame t of a sequence:
  1. tracker.predict(t, camera motion t-1 -> t)                (motion cache; None = no estimate)
  2. main candidates: every cached main ball at conf >= the tracker's main floor (up to 5)
  3. the specialist runs when the tracker has a track whose last measurement is at most max_miss
     frames old AND main's best candidate is below spec_run_below (0.15 = "main missed", today's
     rule). Window centre: the last measurement (`last_seen`, today) or the tracker's prediction
     (`predicted`). n_windows 3 adds windows shifted +-half a crop along the predicted velocity
     (or horizontally when it is ~0) from gap >= 2; the highest peak wins.
  4. tracker.update(candidates) -> this frame's live output (main | specialist | predicted).

Batch on top: ball_tracker.best_path over the same per-frame candidates, then interpolate.

Every output is scored with eval_ball_temporal.score (15 px centre match), so numbers compare
directly with gate 3. Sources map to score's `src`: specialist -> "spec" (so spec_precision means
the same thing), predicted -> "pred", interpolated -> "interp".
"""

from __future__ import annotations

import sys
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Optional

import numpy as np

ROOT = Path(__file__).resolve().parents[3]
for p in (ROOT / "scripts" / "ball_temporal", ROOT / "scripts" / "eval" / "ball_det", ROOT / "gsr_pipeline"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import ball_tracker as bt  # noqa: E402  (plain module import: no gsr_pipeline package init)
from build_ball_temporal_dataset import CROP, clamp_window  # noqa: E402
from eval_ball_temporal import score  # noqa: E402


@dataclass
class SimConfig:
    tracker: bt.TrackerConfig = field(default_factory=bt.TrackerConfig)
    window_centre: str = "last_seen"      # last_seen | predicted
    n_windows: int = 1                    # 1 | 3 (from gap >= 2)
    spec_run_below: float = 0.15          # run the specialist when main's best candidate is below this
    max_miss: int = 15

    def describe(self) -> str:
        t = self.tracker
        return (f"centre={self.window_centre} windows={self.n_windows} run_below={self.spec_run_below} "
                f"lam={t.lam} coast={t.max_coast_frames} spec_floor={t.min_conf_spec} main_floor={t.min_conf_main} "
                f"gate={t.gate_a}+{t.gate_b}g")


def _windows(cx, cy, vel, gap, cfg: SimConfig, W, H):
    tops = [clamp_window(cx, cy, W, H)[:2]]
    if cfg.n_windows >= 3 and gap >= 2:
        vx, vy = vel
        n = float(np.hypot(vx, vy))
        ux, uy = (vx / n, vy / n) if n > 1.0 else (1.0, 0.0)
        for s in (-1, 1):
            tops.append(clamp_window(cx + s * ux * CROP / 2, cy + s * uy * CROP / 2, W, H)[:2])
    return list(dict.fromkeys(tops))       # dedupe windows clamped to the same place


def simulate_tracked(sd, spec, motion: dict, cfg: SimConfig) -> list[dict]:
    """-> per frame {t, live: (x, y, src) | None, cands: [BallCandidate], gap, spec_ran}."""
    tr = bt.BallTracker(cfg.tracker)
    out = []
    for t in sorted(sd.files):
        pred = tr.predict(t, motion.get(t))
        cands = [bt.BallCandidate(b[0], b[1], b[4], "main", b[2], b[3])
                 for b in sd.dets.get(t, []) if b[4] >= cfg.tracker.min_conf_main]
        main_best = max((c.conf for c in cands), default=0.0)
        gap = tr.gap if tr.last_meas is not None else None
        ran = False
        if spec is not None and tr.active and gap is not None and 1 <= gap <= cfg.max_miss \
                and main_best < cfg.spec_run_below:
            if cfg.window_centre == "predicted" and pred is not None:
                cx, cy = pred
            else:
                cx, cy = tr.last_meas[0], tr.last_meas[1]
            vel = (float(tr.x[2]), float(tr.x[3])) if tr.x is not None else (0.0, 0.0)
            best = None
            for x0, y0 in _windows(cx, cy, vel, gap, cfg, sd.W, sd.H):
                key = (sd.ref.name, t, x0, y0)
                frames = None if key in spec.memo else sd.window_frames(t)
                p = spec(key, frames, x0, y0)
                if best is None or p[2] > best[2]:
                    best = p
            ran = True
            cands.append(bt.BallCandidate(best[0], best[1], best[2], "specialist"))
        est = tr.update(cands)
        live = None
        if est is not None:
            live = (est.x, est.y, {"main": "main", "specialist": "spec", "predicted": "pred"}[est.source])
        out.append(dict(t=t, live=live, cands=cands, gap=gap, spec_ran=ran))
    return out


def _records(sim, mode: str, path_cfg: Optional[bt.PathConfig] = None, motion=None, max_interp=25):
    """Per-frame records in eval_ball_temporal.score's format for one output mode:
    measured | live (measured + coasted) | batch (best_path picks) | batch_interp."""
    recs = []
    if mode in ("measured", "live"):
        for r in sim:
            x = y = None
            src = ""
            if r["live"] is not None and (mode == "live" or r["live"][2] != "pred"):
                x, y, src = r["live"]
            recs.append(dict(t=r["t"], x=x, y=y, src=src, gap=r["gap"] if r["spec_ran"] else None))
        return recs
    ts = [r["t"] for r in sim]
    affines = [None] + [motion.get(t) if motion else None for t in ts[1:]]
    picks = bt.best_path([r["cands"] for r in sim], affines, path_cfg)
    pos = bt.interpolate(picks, affines, max_interp) if mode == "batch_interp" else \
        [(p.x, p.y) if p is not None else None for p in picks]
    for r, p, q in zip(sim, picks, pos):
        if q is None:
            recs.append(dict(t=r["t"], x=None, y=None, src="", gap=None))
        else:
            src = "interp" if p is None else ("spec" if p.source == "specialist" else "main")
            recs.append(dict(t=r["t"], x=q[0], y=q[1], src=src, gap=r["gap"] if r["spec_ran"] else None))
    return recs


def score_modes(sds, sims, motions, path_cfg=None, modes=("measured", "live", "batch", "batch_interp")):
    """{mode: score dict} over a split."""
    res = {}
    for m in modes:
        recs = [_records(sim, m, path_cfg, mo) for sim, mo in zip(sims, motions)]
        res[m] = score(sds, recs)
    return res


def run_split(sds, spec, motions, cfg: SimConfig):
    return [simulate_tracked(sd, spec, mo, cfg) for sd, mo in zip(sds, motions)]


def variant(cfg: SimConfig, **kw) -> SimConfig:
    """Copy of cfg with SimConfig fields or tracker fields (prefixed t_) replaced."""
    tk = {k[2:]: v for k, v in kw.items() if k.startswith("t_")}
    sk = {k: v for k, v in kw.items() if not k.startswith("t_")}
    return replace(cfg, tracker=replace(cfg.tracker, **tk), **sk)


# --------------------------------------------------------------------------- the `track` stage

# Coordinate search on `select` (plan §3.7): each tracker knob in turn, keeping the best, twice.
TRACKER_GRID = {
    "t_min_conf_main": [0.05, 0.10, 0.15],
    "t_min_conf_spec": [0.30, 0.50, 0.70],
    "t_lam": [0.25, 0.5, 1.0],
    "t_max_coast_frames": [0, 5, 10],
    "t_gate_b": [20.0, 30.0, 45.0],
    "t_restart_conf": [0.4, 0.6],
}
# Item 3 window variants (plan §4.1-4.2), a full grid on top of the best tracker.
WINDOW_GRID = [dict(window_centre=c, n_windows=n, spec_run_below=b)
               for c in ("last_seen", "predicted") for n in (1, 3) for b in (0.15, 0.30)]
PATH_GRID = [dict(start_pen=s, c0=c) for s in (0.2, 0.4, 0.8) for c in (0.05, 0.10, 0.20)]


def _objective(res: dict, mode: str, floor: float) -> tuple:
    """Higher is better: meet the precision floor first, then recall."""
    r = res[mode]
    return (r["precision"] >= floor, r["recall"] if r["precision"] >= floor else r["precision"])


def _cfg_dict(cfg: SimConfig, path: Optional[bt.PathConfig]) -> dict:
    from dataclasses import asdict
    d = {"sim": {k: v for k, v in asdict(cfg).items() if k != "tracker"}, "tracker": asdict(cfg.tracker)}
    if path is not None:
        d["path"] = {k: v for k, v in asdict(path).items() if k != "tracker"}
    return d


def stage_track(a, model, channels, dev):
    """Item 2 (+ item 3's window variants): tune on `select`, freeze, report `test`. Writes
    <out>/report_track.md, results_track.json, tracker_config.json, search_log.csv."""
    import csv
    import json
    import time

    import camera_motion_cache as cmc
    from eval_ball_temporal import SeqData, Specialist, discover, simulate

    spec = Specialist(model, channels, dev)
    splits = {}
    for split in ("select", "test"):
        refs = [r for r in discover(a.src) if r.split == split]
        if a.max_seqs:
            refs = refs[:a.max_seqs]
        sds = [SeqData(r, a.cache) for r in refs]
        motions = [cmc.load(a.motion, r) if a.motion else {} for r in refs]
        splits[split] = (sds, motions)
        print(f"[track] {split}: {len(sds)} sequences", flush=True)

    def baseline(split):
        sds, _ = splits[split]
        main = score(sds, [simulate(sd, None, 1.0, a) for sd in sds])
        c5c = score(sds, [simulate(sd, spec, a.system_thr, a) for sd in sds])
        return main, c5c

    sel_main, sel_c5c = baseline("select")
    floor = sel_c5c["precision"]
    print(f"[track] select baseline: main {sel_main['recall']:.2%}/{sel_main['precision']:.2%}, "
          f"main+c5c (thr {a.system_thr}) {sel_c5c['recall']:.2%}/{sel_c5c['precision']:.2%}; "
          f"precision floor {floor:.2%}", flush=True)

    log_rows = []

    def evaluate(cfg, path=None, tag=""):
        t0 = time.time()
        sds, motions = splits["select"]
        sims = run_split(sds, spec, motions, cfg)
        res = score_modes(sds, sims, motions, path)
        row = {"tag": tag, "config": cfg.describe(), **{f"{m}_{k}": round(res[m][k], 4)
               for m in res for k in ("recall", "precision")}, "spec_calls": spec.calls,
               "seconds": round(time.time() - t0, 1)}
        log_rows.append(row)
        print(f"[track] {tag:<10} {cfg.describe()} | live {res['live']['recall']:.2%}/{res['live']['precision']:.2%} "
              f"measured {res['measured']['recall']:.2%}/{res['measured']['precision']:.2%} "
              f"batch+interp {res['batch_interp']['recall']:.2%}/{res['batch_interp']['precision']:.2%} "
              f"({row['seconds']}s)", flush=True)
        return res, sims

    best_cfg = SimConfig()
    best_res, best_sims = evaluate(best_cfg, tag="start")
    for p in range(a.search_passes):
        for knob, values in TRACKER_GRID.items():
            for v in values:
                cand = variant(best_cfg, **{knob: v})
                if cand == best_cfg:
                    continue
                res, sims = evaluate(cand, tag=f"p{p + 1} {knob[2:]}")
                if _objective(res, "live", floor) > _objective(best_res, "live", floor):
                    best_cfg, best_res, best_sims = cand, res, sims
    tracker_only = best_cfg
    for w in WINDOW_GRID:
        cand = variant(best_cfg, **w)
        if cand == best_cfg:
            continue
        res, sims = evaluate(cand, tag="window")
        if _objective(res, "live", floor) > _objective(best_res, "live", floor):
            best_cfg, best_res, best_sims = cand, res, sims

    sds, motions = splits["select"]
    best_path_cfg = bt.PathConfig(tracker=best_cfg.tracker)
    best_path_res = score_modes(sds, best_sims, motions, best_path_cfg, ("batch_interp",))
    for g in PATH_GRID:
        pc = bt.PathConfig(tracker=best_cfg.tracker, **g)
        res = score_modes(sds, best_sims, motions, pc, ("batch_interp",))
        print(f"[track] path {g} | batch+interp {res['batch_interp']['recall']:.2%}/"
              f"{res['batch_interp']['precision']:.2%}", flush=True)
        if _objective(res, "batch_interp", floor) > _objective(best_path_res, "batch_interp", floor):
            best_path_cfg, best_path_res = pc, res

    print(f"[track] chosen on select: {best_cfg.describe()}; path start_pen={best_path_cfg.start_pen} "
          f"c0={best_path_cfg.c0}", flush=True)

    # ---- frozen, on test
    test_main, test_c5c = baseline("test")
    sds, motions = splits["test"]
    spec.calls, spec.ms = 0, 0.0
    sims = run_split(sds, spec, motions, best_cfg)
    test = score_modes(sds, sims, motions, best_path_cfg)
    calls, ms = spec.calls, spec.ms / max(spec.calls, 1)
    sims_t = run_split(sds, spec, motions, tracker_only) if tracker_only != best_cfg else sims
    test_tracker_only = score_modes(sds, sims_t, motions, best_path_cfg, ("live",))["live"]

    live = test["live"]
    gate_t = (live["recall"] > test_c5c["recall"] and live["precision"] >= test_c5c["precision"]
              and live["longest_miss_mean_s"] < test_c5c["longest_miss_mean_s"])

    def row(name, r):
        return (f"| {name} | {r['recall']:.2%} | {r['precision']:.2%} | {r['outputs_unlabelled']} | "
                f"{r['longest_miss_mean_s']:.1f} s / {r['longest_miss_max_s']:.1f} s |")

    L = ["# Ball tracker: test results (plan item 2, item 3 window variants)", "",
         f"Specialist `{a.ckpt}`, main cache `{a.cache}`, motion cache `{a.motion}`. Every setting chosen on "
         f"`select` ({len(splits['select'][0])} sequences), frozen for `test` ({len(sds)} sequences). 15 px centre match. "
         f"Precision floor for the search: main + c5c's precision on `select`, {floor:.2%}.", "",
         f"Chosen: `{best_cfg.describe()}`; batch path start_pen {best_path_cfg.start_pen}, c0 {best_path_cfg.c0}.", "",
         "| test | recall | precision | outputs on unlabelled frames | longest miss mean / max |", "|---|---|---|---|---|",
         row("main detector alone", test_main), row(f"main + c5c, as wired (thr {a.system_thr})", test_c5c),
         row("tracker, measured only", test["measured"]), row("tracker, live (measured + coasted)", live),
         row("tracker only (no item 3 window change), live", test_tracker_only),
         row("batch path", test["batch"]), row("batch path + interpolation", test["batch_interp"]), "",
         f"Gate T (live recall above main + c5c, precision no lower, longest miss shorter): "
         f"**{'PASS' if gate_t else 'FAIL'}**.", "",
         "Live recall by frames since the last measurement (frames where the specialist ran):", "",
         "| gap | frames | recall |", "|---|---|---|"]
    for k, (r, n) in live["by_gap"].items():
        L.append(f"| {k} | {n} | {r:.2%} |")
    L += ["", f"Specialist calls on test: {calls} ({ms:.1f} ms per call, PyTorch, this machine).", "",
          f"Search log: `search_log.csv` ({len(log_rows)} select runs)."]
    text = "\n".join(L)
    print("\n" + text, flush=True)
    a.out.mkdir(parents=True, exist_ok=True)
    (a.out / "report_track.md").write_text(text + "\n", encoding="utf-8")
    (a.out / "tracker_config.json").write_text(json.dumps(_cfg_dict(best_cfg, best_path_cfg), indent=1))
    (a.out / "results_track.json").write_text(json.dumps(
        {"select_main": sel_main, "select_c5c": sel_c5c, "select_best": best_res, "test_main": test_main,
         "test_c5c": test_c5c, "test": test, "test_tracker_only_live": test_tracker_only, "gate_t": gate_t,
         "config": _cfg_dict(best_cfg, best_path_cfg), "spec_calls_test": calls, "ms_per_call": ms},
        indent=1, default=str))
    with open(a.out / "search_log.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(log_rows[0]))
        w.writeheader()
        w.writerows(log_rows)
    for sds_, _ in splits.values():
        for sd in sds_:
            sd.reader.close()
    return gate_t

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/eval/ball_det/diagnose_ball_misses.py
"""
diagnose_ball_misses.py -- item 1 of docs/plans/BALL_IMPROVEMENT_PLAN.md (§2): where do the
balls that main detector + c5c miss go, and what could each later item recover?

Standalone apart from scripts/ball_temporal/ and eval_ball_temporal.py. Needs torch (the
specialist replay), numpy, opencv-python. Reads the main-detector cache and the camera-motion
cache (camera_motion_cache.py).

  run     replay main + c5c exactly as wired (eval_ball_temporal's fused chain, threshold
          --system_thr) on `select` and `test`, recording every frame's output, every specialist
          call (window, raw confidence), then tag misses, measure displacement, estimate the
          ceiling, and write the audit sheets. GPU recommended; --no_spec replays main only.

Outputs (<out>/):
  frames.jsonl            one row per frame: split, seq, t, gt, out, src, spec window/conf, gap
  misses.csv              one row per miss: every cause flag, the primary cause, ball size/speed
  displacement.csv        per gap 1-25: p50/p90/p99 ball displacement, raw and camera-compensated
  report_diagnosis.md     cause tables (select, test), displacement, ceiling per item, FP analysis
  diagnosis.json          the same numbers, machine-readable (the orchestrator reads it)
  audit/page_XX.jpg       200 sampled test misses, 20 per page, causes drawn on
  audit/audit.csv         the same 200, with an empty `human_cause` column for the manual pass

Cause rules (a miss = ball-labelled frame with no output within 15 px), primary = first match in:
  near_miss       an output within 15-30 px of a GT ball
  low_conf        a cached main ball within 15 px at conf in [cache floor, --main_conf)
  occluded        GT ball centre inside a GT person box expanded by 10%
  outside_window  the specialist ran but the GT ball was outside its 640x640 window
  not_attempted   the specialist didn't run (no sighting yet, or gap > max_miss)
  fast            GT ball moved > 25 px since the previous frame (image pixels: blur)
  small           GT ball box max side < 10 px
  static_restart  GT ball inside a run of >= 25 frames moving < 1 px/frame (kickoff, set pieces)
  detector_blind  none of the above
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[2]
for p in (ROOT / "scripts" / "ball_temporal", HERE):
    sys.path.insert(0, str(p))
import camera_motion_cache as cmc  # noqa: E402
from build_ball_temporal_dataset import CROP, clamp_window  # noqa: E402
from gsr_io import discover, parse_labels  # noqa: E402

MATCH_PX, NEAR_PX = 15.0, 30.0
FAST_PX, SMALL_PX = 25.0, 10.0
STATIC_PX, STATIC_RUN = 1.0, 25
CAUSES = ("near_miss", "low_conf", "occluded", "outside_window", "not_attempted", "fast", "small",
          "static_restart", "detector_blind")
ITEM_OF = {"low_conf": "item 2 (tracker)", "not_attempted": "item 2 (tracker)",
           "outside_window": "item 3 (c5c windows)", "fast": "item 4 (detector)", "small": "item 4 (detector)",
           "static_restart": "item 4 (detector)", "detector_blind": "item 4 (detector)",
           "near_miss": "localisation", "occluded": "batch interpolation only"}


def replay(sd, spec, thr, a):
    """eval_ball_temporal.simulate (fused chain), recording the specialist's window and raw conf."""
    out, last = [], None
    for t in sorted(sd.files):
        m = [b for b in sd.dets.get(t, []) if b[4] >= a.main_conf]
        if m:
            last = (m[0][0], m[0][1], t)
            out.append(dict(t=t, x=m[0][0], y=m[0][1], src="main", gap=None, win=None, sconf=None))
            continue
        rec = dict(t=t, x=None, y=None, src="", gap=None, win=None, sconf=None)
        if spec is not None and last is not None and t - last[2] <= a.max_miss:
            x0, y0, _ = clamp_window(last[0], last[1], sd.W, sd.H)
            rec["gap"] = t - last[2]
            key = (sd.ref.name, t, x0, y0)
            frames = None if key in spec.memo else sd.window_frames(t)
            px, py, conf = spec(key, frames, x0, y0)
            rec.update(win=(x0, y0), sconf=float(conf))
            if conf >= thr:
                rec.update(x=px, y=py, src="spec")
                last = (px, py, t)
        out.append(rec)
    return out


def _static_frames(gt_pos: dict) -> set:
    ts = sorted(gt_pos)
    still = [t for t, u in zip(ts[1:], ts) if t == u + 1 and
             np.hypot(gt_pos[t][0] - gt_pos[u][0], gt_pos[t][1] - gt_pos[u][1]) < STATIC_PX]
    runs, cur = set(), []
    for t in still:
        if cur and t != cur[-1] + 1:
            if len(cur) >= STATIC_RUN:
                runs.update(cur)
            cur = []
        cur.append(t)
    if len(cur) >= STATIC_RUN:
        runs.update(cur)
    return runs


def _off_pitch(p, line_pts) -> bool:
    if len(line_pts) < 3:
        return False
    import cv2
    hull = cv2.convexHull(np.array(line_pts, np.float32))
    return cv2.pointPolygonTest(hull, (float(p[0]), float(p[1])), False) < 0


def _displacements(gt_pos, motion, max_gap=25):
    """{gap: ([raw], [camera-compensated])} over pairs of labelled frames."""
    out = defaultdict(lambda: ([], []))
    for t in sorted(gt_pos):
        m = np.eye(3)
        ok = True
        for g in range(1, max_gap + 1):
            u = t + g
            a = motion.get(u) if motion else None
            if a is None:
                ok = False
            elif ok:
                m = np.vstack([a, [0, 0, 1]]) @ m
            if u not in gt_pos:
                continue
            p, q = gt_pos[t], gt_pos[u]
            out[g][0].append(float(np.hypot(q[0] - p[0], q[1] - p[1])))
            if ok:
                px = m[0, 0] * p[0] + m[0, 1] * p[1] + m[0, 2]
                py = m[1, 0] * p[0] + m[1, 1] * p[1] + m[1, 2]
                out[g][1].append(float(np.hypot(q[0] - px, q[1] - py)))
    return out


def analyse_sequence(split, sd, out, labels, motion, cache_floor, main_conf):
    _im, _files, balls, persons, lines, _wh = labels
    gt_pos = {t: (bs[0][0], bs[0][1]) for t, bs in balls.items() if bs}
    static = _static_frames(gt_pos)
    frames, misses, fps_ = [], [], []
    for r in out:
        t = r["t"]
        gts = balls.get(t, [])
        o = (r["x"], r["y"]) if r["x"] is not None else None
        d_out = min((np.hypot(o[0] - g[0], o[1] - g[1]) for g in gts), default=np.inf) if o else np.inf
        hit = bool(gts) and d_out <= MATCH_PX
        frames.append(dict(split=split, seq=sd.ref.name, t=t, gt=[(g[0], g[1]) for g in gts], out=o, src=r["src"],
                           win=r["win"], sconf=r["sconf"], gap=r["gap"], hit=hit))
        if gts and o is not None and not hit:
            fps_.append(dict(dist=float(d_out), offpitch=_off_pitch(o, lines.get(t, [])), src=r["src"]))
        if not gts or hit:
            continue
        g = gts[0]
        flags = {c: False for c in CAUSES}
        flags["near_miss"] = d_out <= NEAR_PX
        flags["low_conf"] = any(cache_floor <= b[4] < main_conf and np.hypot(b[0] - g[0], b[1] - g[1]) <= MATCH_PX
                                for b in sd.dets.get(t, []))
        flags["occluded"] = any(abs(g[0] - p[0]) <= 0.55 * p[2] and abs(g[1] - p[1]) <= 0.55 * p[3]
                                for p in persons.get(t, []))
        if r["win"] is not None:
            x0, y0 = r["win"]
            flags["outside_window"] = not (x0 <= g[0] < x0 + CROP and y0 <= g[1] < y0 + CROP)
        else:
            flags["not_attempted"] = True
        prev = gt_pos.get(t - 1)
        speed = float(np.hypot(g[0] - prev[0], g[1] - prev[1])) if prev else 0.0
        flags["fast"] = speed > FAST_PX
        flags["small"] = max(g[2], g[3]) < SMALL_PX
        flags["static_restart"] = t in static
        primary = next((c for c in CAUSES[:-1] if flags[c]), "detector_blind")
        flags["detector_blind"] = primary == "detector_blind"
        misses.append(dict(split=split, seq=sd.ref.name, t=t, primary=primary, gx=round(g[0], 1), gy=round(g[1], 1),
                           size=round(max(g[2], g[3]), 1), speed=round(speed, 1), gap=r["gap"] if r["gap"] else "",
                           sconf=round(r["sconf"], 3) if r["sconf"] is not None else "",
                           **{c: int(v) for c, v in flags.items()}))
    return frames, misses, fps_, _displacements(gt_pos, motion)


def cause_table(misses, n_gt, recall):
    c = Counter(m["primary"] for m in misses)
    flag = {k: sum(m[k] for m in misses) for k in CAUSES}
    n = max(len(misses), 1)
    rows = ["| primary cause | misses | share of misses | share of ball frames | flagged (any) | addressed by |",
            "|---|---|---|---|---|---|"]
    for k in CAUSES:
        rows.append(f"| {k} | {c[k]} | {c[k] / n:.1%} | {c[k] / max(n_gt, 1):.1%} | {flag[k]} | {ITEM_OF[k]} |")
    rows.append(f"| **total** | {len(misses)} | 100% | {len(misses) / max(n_gt, 1):.1%} (recall {recall:.1%}) | | |")
    return rows, dict(c)


def ceiling(primary_counts, n_gt, recall):
    share = {k: v / max(n_gt, 1) for k, v in primary_counts.items()}
    it2 = share.get("low_conf", 0) + share.get("not_attempted", 0)
    it3 = share.get("outside_window", 0)
    it4 = sum(share.get(k, 0) for k in ("fast", "small", "static_restart", "detector_blind"))
    occ = share.get("occluded", 0)
    loc = share.get("near_miss", 0)
    return {"now": recall, "item2": it2, "item3": it3, "item4": it4, "occluded": occ, "near_miss": loc,
            "live_ceiling": recall + it2 + it3 + it4 + loc, "batch_ceiling": recall + it2 + it3 + it4 + loc + occ}


def audit_sheets(misses, sds_by_name, out_dir: Path, frames_by_key, n=200, seed=0, per_page=20):
    import cv2
    out_dir.mkdir(parents=True, exist_ok=True)
    pool = [m for m in misses if m["split"] == "test"]
    if not pool:
        return 0
    rng = np.random.default_rng(seed)
    pick = [pool[i] for i in sorted(rng.choice(len(pool), min(n, len(pool)), replace=False))]
    tiles = []
    with open(out_dir / "audit.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["page", "slot", "seq", "t", "primary", "flags", "human_cause", "notes"])
        w.writeheader()
        for i, m in enumerate(pick):
            sd = sds_by_name[m["seq"]]
            img = sd.frame(m["t"]).copy()
            x0, y0, _ = clamp_window(m["gx"], m["gy"], sd.W, sd.H)
            fr = frames_by_key[(m["seq"], m["t"])]
            if fr["win"] is not None:
                wx, wy = fr["win"]
                cv2.rectangle(img, (int(wx), int(wy)), (int(wx) + CROP, int(wy) + CROP), (0, 255, 255), 3)
            cv2.circle(img, (int(m["gx"]), int(m["gy"])), int(max(m["size"], 8)) + 6, (0, 255, 0), 2)
            if fr["out"] is not None:
                cv2.drawMarker(img, (int(fr["out"][0]), int(fr["out"][1])), (0, 0, 255), cv2.MARKER_CROSS, 24, 2)
            crop = img[y0:y0 + CROP, x0:x0 + CROP].copy()
            flags = ",".join(k for k in CAUSES if m[k])
            for txt, yy in ((f"#{i} {m['seq']} f{m['t']} {m['primary']}", 30), (flags[:60], 62)):
                cv2.putText(crop, txt, (8, yy), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
                cv2.putText(crop, txt, (8, yy), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            tiles.append(cv2.resize(crop, (320, 320), interpolation=cv2.INTER_AREA))
            w.writerow(dict(page=i // per_page, slot=i % per_page, seq=m["seq"], t=m["t"], primary=m["primary"],
                            flags=flags, human_cause="", notes=""))
    for p in range(0, len(tiles), per_page):
        page = tiles[p:p + per_page]
        page += [np.zeros_like(tiles[0])] * (per_page - len(page))
        grid = np.concatenate([np.concatenate(page[r * 5:(r + 1) * 5], axis=1) for r in range(per_page // 5)], axis=0)
        cv2.imwrite(str(out_dir / f"page_{p // per_page:02d}.jpg"), grid, [cv2.IMWRITE_JPEG_QUALITY, 88])
    return len(pick)


def stage_run(a):
    import torch
    from eval_ball_temporal import SeqData, Specialist, load_model
    spec = None
    if not a.no_spec:
        model, channels, epoch = load_model(a.ckpt, torch.device(a.device))
        spec = Specialist(model, channels, torch.device(a.device))
        print(f"[diagnose] specialist {a.ckpt} (epoch {epoch}), threshold {a.system_thr}", flush=True)
    a.out.mkdir(parents=True, exist_ok=True)
    report, data = [], {"splits": {}}
    all_frames, all_misses, sds_by_name, frames_by_key = [], [], {}, {}
    disp_all = defaultdict(lambda: ([], []))
    for split in a.splits:
        refs = [r for r in discover(a.src) if r.split == split]
        if a.max_seqs:
            refs = refs[:a.max_seqs]
        misses_s, fps_s, n_gt, hits = [], [], 0, 0
        t0 = time.time()
        for i, ref in enumerate(refs, 1):
            sd = SeqData(ref, a.cache)
            meta = json.loads((a.cache / ref.split / f"{ref.name}.json").read_text()).get("meta", {})
            cache_floor = float(meta.get("conf", 0.05))
            has_motion = a.motion and (a.motion / ref.split / f"{ref.name}.json").exists()
            motion = cmc.load(a.motion, ref) if has_motion else {}
            out = replay(sd, spec, a.system_thr, a)
            labels = parse_labels(sd.reader.labels())
            frames, misses, fps_, disp = analyse_sequence(split, sd, out, labels, motion, cache_floor, a.main_conf)
            for g, (raw, comp) in disp.items():
                disp_all[g][0].extend(raw)
                disp_all[g][1].extend(comp)
            all_frames += frames
            misses_s += misses
            fps_s += fps_
            sds_by_name[ref.name] = sd
            for fr in frames:
                frames_by_key[(fr["seq"], fr["t"])] = fr
            seq_gt = sum(bool(f["gt"]) for f in frames)
            seq_hit = sum(f["hit"] for f in frames)
            n_gt, hits = n_gt + seq_gt, hits + seq_hit
            print(f"[diagnose {split} {i}/{len(refs)}] {ref.name}: recall {seq_hit / max(seq_gt, 1):.1%}, "
                  f"{len(misses)} misses, top causes {Counter(m['primary'] for m in misses).most_common(3)} "
                  f"({time.time() - t0:.0f}s)", flush=True)
        recall = hits / max(n_gt, 1)
        rows, prim = cause_table(misses_s, n_gt, recall)
        ceil = ceiling(prim, n_gt, recall)
        offp = sum(f["offpitch"] for f in fps_s)
        fp_med = float(np.median([f["dist"] for f in fps_s])) if fps_s else None
        data["splits"][split] = {"sequences": len(refs), "ball_frames": n_gt, "recall": recall,
                                 "misses": len(misses_s), "primary": prim, "ceiling": ceil,
                                 "false_positives": len(fps_s), "fp_offpitch": offp, "fp_median_dist": fp_med}
        report += [f"## `{split}`: {len(refs)} sequences, {n_gt} ball frames, main + c5c recall {recall:.2%}", ""] + rows + [
            "", f"Ceiling if every miss of a kind were fixed (shares of ball frames): item 2 +{ceil['item2']:.1%}, "
                f"item 3 +{ceil['item3']:.1%}, item 4 +{ceil['item4']:.1%}, localisation +{ceil['near_miss']:.1%}; "
                f"hidden behind players {ceil['occluded']:.1%} (batch interpolation only). "
                f"**Live ceiling {ceil['live_ceiling']:.1%}, batch ceiling {ceil['batch_ceiling']:.1%}.**", "",
            f"False positives on ball-labelled frames: {len(fps_s)}, of which {offp} outside the pitch-line hull "
            f"(likely a spare ball); median distance to the labelled ball {fp_med or 0:.0f} px.", ""]
        all_misses += misses_s

    disp_rows = []
    for g in sorted(disp_all):
        raw = np.array(disp_all[g][0]) if disp_all[g][0] else np.array([np.nan])
        comp = np.array(disp_all[g][1]) if disp_all[g][1] else np.array([np.nan])
        disp_rows.append({"gap": g, "n": len(disp_all[g][0]),
                          "raw_p50": float(np.nanpercentile(raw, 50)), "raw_p90": float(np.nanpercentile(raw, 90)),
                          "raw_p99": float(np.nanpercentile(raw, 99)), "comp_n": len(disp_all[g][1]),
                          "comp_p50": float(np.nanpercentile(comp, 50)), "comp_p90": float(np.nanpercentile(comp, 90)),
                          "comp_p99": float(np.nanpercentile(comp, 99))})
    fit = None
    pts = [(r["gap"], r["comp_p99"]) for r in disp_rows if r["comp_n"] >= 20 and np.isfinite(r["comp_p99"])]
    if len(pts) >= 3:
        gg, vv = np.array(pts).T
        b, a0 = np.polyfit(gg, vv, 1)
        fit = {"gate_a": float(max(a0, 10.0)), "gate_b": float(max(b, 1.0))}
    data["displacement"] = disp_rows
    data["gate_fit"] = fit
    report += ["## Ball displacement between labelled frames (all diagnosed splits)", "",
               "| gap | pairs | raw p50 / p90 / p99 px | camera-compensated pairs | compensated p50 / p90 / p99 px |",
               "|---|---|---|---|---|"]
    for r in disp_rows:
        report.append(f"| {r['gap']} | {r['n']} | {r['raw_p50']:.0f} / {r['raw_p90']:.0f} / {r['raw_p99']:.0f} | "
                      f"{r['comp_n']} | {r['comp_p50']:.0f} / {r['comp_p90']:.0f} / {r['comp_p99']:.0f} |")
    report += ["", f"Gate fit to the compensated p99 (r(g) = a + b*g): {fit}" if fit else
               "Gate fit: not enough camera-compensated pairs (no motion cache?)", ""]

    n_audit = audit_sheets(all_misses, sds_by_name, a.out / "audit", frames_by_key, n=a.audit, seed=a.seed) if a.audit else 0
    report += [f"Audit: {n_audit} sampled `test` misses in `audit/page_*.jpg`; fill `audit/audit.csv`'s `human_cause` "
               "column on one pass by eye, then compare with `primary` (plan §2.4: rules below ~80% agreement get "
               "fixed on `select`).", ""]
    spec_txt = "off" if a.no_spec else f"`{a.ckpt}` at {a.system_thr}"
    text = "\n".join(["# Ball miss diagnosis (plan item 1)", "",
                      f"Main cache `{a.cache}` (output conf >= {a.main_conf}), specialist {spec_txt}, "
                      f"motion cache `{a.motion}`. 15 px centre match. A miss can carry several causes; the table "
                      "counts each under its primary (first-matching) cause.", ""] + report)
    print("\n" + text, flush=True)
    (a.out / "report_diagnosis.md").write_text(text + "\n", encoding="utf-8")
    (a.out / "diagnosis.json").write_text(json.dumps(data, indent=1, default=float))
    with open(a.out / "frames.jsonl", "w") as f:
        for fr in all_frames:
            f.write(json.dumps(fr, default=float) + "\n")
    if all_misses:
        with open(a.out / "misses.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(all_misses[0]))
            w.writeheader()
            w.writerows(all_misses)
    with open(a.out / "displacement.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(disp_rows[0]) if disp_rows else ["gap"])
        w.writeheader()
        w.writerows(disp_rows)
    for sd in sds_by_name.values():
        sd.reader.close()


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["run"])
    ap.add_argument("--src", type=Path, required=True)
    ap.add_argument("--cache", type=Path, required=True, help="cache_main_detections.py output")
    ap.add_argument("--motion", type=Path, default=None, help="camera_motion_cache.py output")
    ap.add_argument("--ckpt", type=Path, default=ROOT / "runs" / "ball_temporal" / "c5c" / "best.pt")
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--splits", nargs="+", default=["select", "test"])
    ap.add_argument("--main_conf", type=float, default=0.15)
    ap.add_argument("--max_miss", type=int, default=15)
    ap.add_argument("--system_thr", type=float, default=0.70)
    ap.add_argument("--no_spec", action="store_true")
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--audit", type=int, default=200, help="sampled test misses for the audit sheets (0 = none)")
    ap.add_argument("--max_seqs", type=int, default=0, help="first N sequences per split (smoke tests)")
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    a.chain = "fused"
    stage_run(a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/gsr_pipeline/ball_tracker.py
"""
ball_tracker.py -- one ball, many candidates: the tracker of docs/plans/BALL_IMPROVEMENT_PLAN.md
§3 (item 2). Pure numpy, no package imports, so the evaluators in scripts/ and the Kaggle kit can
import this file directly.

Live (`BallTracker`), per frame:
  1. `predict(frame_idx, affine)`: move the state by the camera motion (a 2x3 affine mapping the
     previous frame's pixels into this frame's), then a constant-velocity Kalman prediction.
     Returns the predicted position, which a caller may use to place the specialist's window.
  2. `update(candidates)`: candidates inside the gate radius r(g) = min(gate_a + gate_b*g, gate_max),
     g = frames since the last accepted measurement, compete on cost d/r - lam*conf; main beats
     specialist at equal cost. Accepted -> Kalman update. None -> coast on the prediction for up
     to max_coast_frames (source "predicted"), then drop the track.
     A strong candidate (conf >= restart_conf) outside the gate on restart_frames consecutive
     frames, each within r(1) of the previous, starts a new track (a cut or a long pass).
     With no track, the strongest candidate at conf >= init_conf starts one.

Batch (`best_path`): the whole sequence is known, so pick the ball per frame by dynamic
programming over every frame's candidates (a longest path through a DAG: emission reward per
candidate, transition penalty by distance against r(gap), a start penalty per new segment), then
`interpolate` fills gaps between consecutive picks up to max_interp_frames, compensating camera
motion. (The plan names RTS smoothing too; it is left out until the DP + interpolation result
shows positions, not picks, are what's wrong.)

Confidences: candidate.conf is the model's own. Specialist confidences are multiplied by
spec_scale before any comparison, so the two sources can be put on one scale (fit on `select`).
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional, Sequence

import numpy as np


@dataclass(frozen=True)
class BallCandidate:
    x: float
    y: float
    conf: float
    source: str = "main"           # "main" | "specialist"
    w: float = 0.0
    h: float = 0.0


@dataclass(frozen=True)
class BallEstimate:
    x: float
    y: float
    conf: float
    source: str                    # "main" | "specialist" | "predicted"
    track_id: int
    frames_since_seen: int


@dataclass
class TrackerConfig:
    gate_a: float = 40.0           # px: gate radius at gap 0 ...
    gate_b: float = 30.0           # ... plus this much per frame since the last measurement
    gate_max: float = 500.0
    lam: float = 0.5               # weight of confidence against normalised distance
    min_conf_main: float = 0.05    # candidate floors per source
    min_conf_spec: float = 0.30
    spec_scale: float = 1.0        # specialist confidence multiplier (put it on main's scale)
    init_conf: float = 0.15        # start a track (no track yet) from a candidate this strong
    restart_conf: float = 0.50     # start a new track from a strong candidate outside the gate ...
    restart_frames: int = 2        # ... seen on this many consecutive frames
    max_coast_frames: int = 10     # predicted output after the last measurement, then drop
    q_pos: float = 4.0             # process noise (px^2 per frame) on position ...
    q_vel: float = 9.0             # ... and on velocity
    r_main: float = 4.0            # measurement noise (px^2): detector box centres
    r_spec: float = 9.0            # specialist points
    vel_damping: float = 0.85      # velocity kept per coasted frame (a coast shouldn't fly off)

    def gate(self, gap: int) -> float:
        return min(self.gate_a + self.gate_b * max(gap, 0), self.gate_max)

    def eff_conf(self, c: BallCandidate) -> float:
        return c.conf * (self.spec_scale if c.source == "specialist" else 1.0)

    def floor(self, c: BallCandidate) -> float:
        return self.min_conf_spec if c.source == "specialist" else self.min_conf_main


def apply_affine(affine, x: float, y: float) -> tuple[float, float]:
    if affine is None:
        return x, y
    a = np.asarray(affine, float)
    return float(a[0, 0] * x + a[0, 1] * y + a[0, 2]), float(a[1, 0] * x + a[1, 1] * y + a[1, 2])


def _rotate(affine, vx: float, vy: float) -> tuple[float, float]:
    if affine is None:
        return vx, vy
    a = np.asarray(affine, float)
    return float(a[0, 0] * vx + a[0, 1] * vy), float(a[1, 0] * vx + a[1, 1] * vy)


class BallTracker:
    """Online single-ball tracker. Call predict() then update() once per frame, in frame order."""

    def __init__(self, cfg: Optional[TrackerConfig] = None) -> None:
        self.cfg = cfg or TrackerConfig()
        self.reset()

    def reset(self) -> None:
        self.x: Optional[np.ndarray] = None      # state (x, y, vx, vy)
        self.P: Optional[np.ndarray] = None
        self.track_id = 0
        self.last_meas: Optional[tuple[float, float, int]] = None   # (x, y, frame) of the last accepted
        self._frame: Optional[int] = None
        self._pending: list[tuple[int, float, float]] = []          # strong out-of-gate candidates
        self._coasting = False
        self.stats = {"main": 0, "specialist": 0, "predicted": 0, "restarts": 0, "drops": 0}

    @property
    def active(self) -> bool:
        return self.x is not None

    @property
    def gap(self) -> int:
        """Frames since the last accepted measurement (at the current frame)."""
        if self.last_meas is None or self._frame is None:
            return 10 ** 6
        return self._frame - self.last_meas[2]

    @property
    def position(self) -> Optional[tuple[float, float]]:
        return None if self.x is None else (float(self.x[0]), float(self.x[1]))

    def _start(self, c: BallCandidate, frame_idx: int) -> None:
        self.track_id += 1
        self.x = np.array([c.x, c.y, 0.0, 0.0])
        r = self.cfg.r_spec if c.source == "specialist" else self.cfg.r_main
        self.P = np.diag([r, r, 400.0, 400.0])     # velocity unknown at birth
        self.last_meas = (c.x, c.y, frame_idx)
        self._pending = []
        self._coasting = False

    def predict(self, frame_idx: int, affine=None) -> Optional[tuple[float, float]]:
        """Advance to frame_idx (camera motion, then constant velocity). Returns the prediction."""
        dt = 1 if self._frame is None else max(frame_idx - self._frame, 1)
        self._frame = frame_idx
        if self.last_meas is not None and affine is not None:
            lx, ly = apply_affine(affine, self.last_meas[0], self.last_meas[1])
            self.last_meas = (lx, ly, self.last_meas[2])
        self._pending = [(f, *apply_affine(affine, px, py)) for f, px, py in self._pending]
        if self.x is None:
            return None
        x, y = apply_affine(affine, self.x[0], self.x[1])
        vx, vy = _rotate(affine, self.x[2], self.x[3])
        if self._coasting:
            vx, vy = vx * self.cfg.vel_damping, vy * self.cfg.vel_damping
        F = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], float)
        self.x = F @ np.array([x, y, vx, vy])
        Q = np.diag([self.cfg.q_pos, self.cfg.q_pos, self.cfg.q_vel, self.cfg.q_vel]) * dt
        self.P = F @ self.P @ F.T + Q
        return self.position

    def update(self, candidates: Sequence[BallCandidate]) -> Optional[BallEstimate]:
        """Choose this frame's ball (call after predict). Returns None when there is none."""
        cfg, frame = self.cfg, self._frame
        if frame is None:
            raise RuntimeError("call predict(frame_idx, affine) before update()")
        cands = [c for c in candidates if c.conf >= cfg.floor(c)]

        if self.x is None:
            if cands:
                best = max(cands, key=lambda c: (cfg.eff_conf(c), c.source == "main"))
                if cfg.eff_conf(best) >= cfg.init_conf:
                    self._start(best, frame)
                    self.stats[best.source] += 1
                    return self._estimate(best.conf, best.source, 0)
            return None

        gap = frame - self.last_meas[2]
        r = cfg.gate(gap)
        px, py = self.x[0], self.x[1]
        inside = []
        for c in cands:
            d = float(np.hypot(c.x - px, c.y - py))
            if d <= r:
                inside.append((d / r - cfg.lam * cfg.eff_conf(c), c.source != "main", c))
        if inside:
            _, _, c = min(inside, key=lambda t: (t[0], t[1]))
            self._kalman_update(c)
            self.last_meas = (c.x, c.y, frame)
            self._pending = []
            self._coasting = False
            self.stats[c.source] += 1
            return self._estimate(c.conf, c.source, 0)

        # Nothing in the gate: a run of strong candidates elsewhere restarts the track.
        strong = [c for c in cands if cfg.eff_conf(c) >= cfg.restart_conf]
        if strong:
            s = max(strong, key=cfg.eff_conf)
            if self._pending and self._pending[-1][0] == frame - 1 and \
                    np.hypot(s.x - self._pending[-1][1], s.y - self._pending[-1][2]) <= cfg.gate(1):
                self._pending.append((frame, s.x, s.y))
            else:
                self._pending = [(frame, s.x, s.y)]
            if len(self._pending) >= cfg.restart_frames:
                self._start(s, frame)
                self.stats["restarts"] += 1
                self.stats[s.source] += 1
                return self._estimate(s.conf, s.source, 0)
        else:
            self._pending = []

        if gap <= cfg.max_coast_frames:
            self._coasting = True
            self.stats["predicted"] += 1
            return self._estimate(0.0, "predicted", gap)
        self.x, self.P = None, None
        self.stats["drops"] += 1
        return None

    def _kalman_update(self, c: BallCandidate) -> None:
        H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], float)
        r = self.cfg.r_spec if c.source == "specialist" else self.cfg.r_main
        S = H @ self.P @ H.T + np.eye(2) * r
        K = self.P @ H.T @ np.linalg.inv(S)
        self.x = self.x + K @ (np.array([c.x, c.y]) - H @ self.x)
        self.P = (np.eye(4) - K @ H) @ self.P

    def _estimate(self, conf: float, source: str, gap: int) -> BallEstimate:
        return BallEstimate(float(self.x[0]), float(self.x[1]), float(conf), source, self.track_id, gap)


# ---------------------------------------------------------------------- batch

@dataclass
class PathConfig:
    """best_path's costs. Rewards and penalties are in confidence units."""
    c0: float = 0.10               # emission reward = eff_conf - c0 (weaker candidates cost)
    w_trans: float = 0.30          # transition penalty = w_trans * (d / r(gap))^2
    skip_pen: float = 0.02         # per frame skipped between two picks
    start_pen: float = 0.30        # per new segment
    max_skip: int = 25             # longest gap a segment may bridge
    tracker: TrackerConfig = field(default_factory=TrackerConfig)


class _Chain:
    """Camera motion over a sequence: compose(s, t) maps frame s pixels to frame t pixels in O(1),
    from cumulative transforms (affines[k]: frame k-1 -> frame k; None = no motion)."""

    def __init__(self, affines: Optional[Sequence], T: int) -> None:
        self.cum = [np.eye(3)]
        for k in range(1, T):
            a = affines[k] if affines is not None and k < len(affines) else None
            step = np.eye(3) if a is None else np.vstack([np.asarray(a, float), [0, 0, 1]])
            self.cum.append(step @ self.cum[-1])
        self.inv = [np.linalg.inv(m) for m in self.cum]

    def compose(self, s: int, t: int) -> np.ndarray:
        return self.cum[t] @ self.inv[s]


def best_path(cands: Sequence[Sequence[BallCandidate]], affines: Optional[Sequence] = None,
              cfg: Optional[PathConfig] = None) -> list[Optional[BallCandidate]]:
    """cands[t]: candidates of frame t (frames 0..T-1, consecutive). affines[t]: frame t-1 -> t
    (None or missing = no motion). Returns the chosen candidate per frame, or None."""
    cfg = cfg or PathConfig()
    tc = cfg.tracker
    T = len(cands)
    chain = _Chain(affines, T)
    nodes = [[c for c in fr if c.conf >= tc.floor(c)] for fr in cands]
    score = [np.full(len(n), -np.inf) for n in nodes]
    back: list[list] = [[None] * len(n) for n in nodes]
    best_upto: list[tuple[float, Optional[tuple[int, int]]]] = [(0.0, None)] * T
    running: tuple[float, Optional[tuple[int, int]]] = (0.0, None)
    for t in range(T):
        for j, c in enumerate(nodes[t]):
            prev_total, prev_node = best_upto[t - 1] if t > 0 else (0.0, None)
            top, arg = prev_total - cfg.start_pen, prev_node       # a new segment after the best so far
            for s in range(max(0, t - cfg.max_skip), t):
                if not nodes[s]:
                    continue
                m = chain.compose(s, t)
                g = t - s
                r = tc.gate(g - 1)
                for i, p in enumerate(nodes[s]):
                    if score[s][i] == -np.inf:
                        continue
                    qx = m[0, 0] * p.x + m[0, 1] * p.y + m[0, 2]
                    qy = m[1, 0] * p.x + m[1, 1] * p.y + m[1, 2]
                    d = float(np.hypot(c.x - qx, c.y - qy))
                    if d > r:
                        continue
                    v = score[s][i] - cfg.w_trans * (d / r) ** 2 - cfg.skip_pen * (g - 1)
                    if v > top:
                        top, arg = v, (s, i)
            score[t][j] = top + tc.eff_conf(c) - cfg.c0
            back[t][j] = arg
            if score[t][j] > running[0]:
                running = (score[t][j], (t, j))
        best_upto[t] = running

    out: list[Optional[BallCandidate]] = [None] * T
    node = running[1]
    while node is not None:
        t, j = node
        out[t] = nodes[t][j]
        node = back[t][j]
    return out


def interpolate(picks: Sequence[Optional[BallCandidate]], affines: Optional[Sequence] = None,
                max_interp_frames: int = 25) -> list[Optional[tuple[float, float]]]:
    """Fill gaps of at most max_interp_frames between consecutive picks, linearly in the earlier
    pick's camera frame, mapped forward by the camera motion. Returns per-frame (x, y) or None;
    picked frames keep their candidate's position."""
    T = len(picks)
    chain = _Chain(affines, T)
    out: list[Optional[tuple[float, float]]] = [(p.x, p.y) if p is not None else None for p in picks]
    idx = [t for t, p in enumerate(picks) if p is not None]
    for s, e in zip(idx, idx[1:]):
        g = e - s
        if g <= 1 or g - 1 > max_interp_frames:
            continue
        inv = chain.compose(e, s)
        ex, ey = picks[e].x, picks[e].y
        bx = inv[0, 0] * ex + inv[0, 1] * ey + inv[0, 2]
        by = inv[1, 0] * ex + inv[1, 1] * ey + inv[1, 2]
        ax_, ay_ = picks[s].x, picks[s].y
        for t in range(s + 1, e):
            f = (t - s) / g
            x, y = ax_ + f * (bx - ax_), ay_ + f * (by - ay_)
            m = chain.compose(s, t)
            out[t] =(m[0, 0] * x + m[0, 1] * y + m[0, 2], m[1, 0] * x + m[1, 1] * y + m[1, 2])
    return out

In [ ]:
%%writefile /kaggle/working/kit/cv_service/notebooks/kaggle/ball_pipeline.py
"""
ball_pipeline.py -- the Kaggle runner for docs/plans/BALL_IMPROVEMENT_PLAN.md items 1-3:
miss diagnosis, ball tracker, and the c5d specialist (train, gate, export), end to end, with
live logs in the notebook cell and a deliverables folder at the end.

The notebook (notebooks/kaggle_ball_improvement.ipynb, generated by
scripts/kaggle/build_ball_notebook.py) writes the kit with %%writefile cells, then calls this
script one stage group per cell. Every stage is resumable: it skips work whose outputs exist, so
re-running a cell after a crash or in a new session continues where it stopped.

Stages (in order; --stages picks a subset, `all` runs every one):
  restore    a new session: copy an earlier version's work folder back from /kaggle/input
             (--restore_from): state, caches, reports, the training run with last.pt. No-op without it.
  env        GPUs, versions, disk; pip-installs what the kit needs (ultralytics>=8.4, onnx, onnxruntime)
  data       copy the GSR dataset and the weights from /kaggle/input to /kaggle/working (or read in
             place when /kaggle/working can't hold it: --copy auto checks free space first)
  cache      main-detector ball cache, every split, both GPUs (or unzip a cache from an earlier run)
  motion     camera-motion cache for select + test (train + pool later, only if c5d needs them)
  diagnose   item 1: miss causes, displacement, ceiling, audit sheets      -> reports/diagnosis/
  track      item 2 (+ item 3 windows): tracker tuned on select, gate T on test -> reports/track_c5c/
  c5d_data   item 3 dataset: window centre chosen by `track`, gap>=2 oversampling
  c5d_train  item 3 training: warm start from c5c, 2-GPU DDP, stops before the session budget
  c5d_eval   gates 1-3 for c5d (new threshold on select) and the tracker re-run with c5d (gate S)
  export     c5d ONNX + sidecar + parity report (batch 1, and batch 3 if the 3-window variant won)
  deliver    /kaggle/working/deliverables/ + deliverables.zip: reports, configs, model files,
             logs, SUMMARY.md, a PERF_LOG entry draft, MANIFEST.json (sha256 of every file)
  cleanup    delete the bulky intermediates (GSR copy, crop dataset) so Save Version fits in 20 GB

Run `python ball_pipeline.py --help` for every path and knob. Kaggle defaults are built in.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from datetime import datetime
from pathlib import Path

STAGES = ["restore", "env", "data", "cache", "motion", "diagnose", "track", "c5d_data", "c5d_train", "c5d_eval",
          "export", "deliver", "cleanup"]
KAGGLE_IN = Path(os.environ.get("BALL_PIPELINE_INPUT", "/kaggle/input"))      # overridable for local smoke tests
WORKING = Path(os.environ.get("BALL_PIPELINE_WORKING", "/kaggle/working"))


# ----------------------------------------------------------------------------- logging

class Log:
    """Timestamped lines to the cell (stdout, flushed) and to pipeline.log / <stage>.log."""

    def __init__(self, log_dir: Path):
        log_dir.mkdir(parents=True, exist_ok=True)
        self.dir = log_dir
        self.main = open(log_dir / "pipeline.log", "a", buffering=1, encoding="utf-8")
        self.stage_f = None
        self.lock = threading.Lock()

    def stage(self, name):
        if self.stage_f:
            self.stage_f.close()
        self.stage_f = open(self.dir / f"{name}.log", "a", buffering=1, encoding="utf-8") if name else None

    def __call__(self, msg: str, tag: str = "pipeline"):
        line = f"{datetime.now():%H:%M:%S} [{tag}] {msg}"
        with self.lock:
            print(line, flush=True)
            self.main.write(line + "\n")
            if self.stage_f:
                self.stage_f.write(line + "\n")

    def banner(self, title: str):
        bar = "=" * 96
        for s in ("", bar, f"  {title}", bar):
            self(s)


def gpu_line() -> str:
    try:
        q = subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total",
                            "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10).stdout
        parts = [l.split(", ") for l in q.strip().splitlines()]
        return " | ".join(f"gpu{i} {u}% {int(m) / 1024:.1f}/{int(t) / 1024:.0f}G" for i, u, m, t in parts)
    except Exception:
        return "no nvidia-smi"


class Heartbeat(threading.Thread):
    """Every `every` s: elapsed stage time, session time left, GPU use, free disk."""

    def __init__(self, log: Log, ctx, every: int = 60):
        super().__init__(daemon=True)
        self.log, self.ctx, self.every = log, ctx, every
        self.stage, self.t0 = None, time.time()
        self._halt = threading.Event()

    def set(self, stage):
        self.stage, self.t0 = stage, time.time()

    def run(self):
        while not self._halt.wait(self.every):
            if self.stage:
                free = shutil.disk_usage(WORKING if WORKING.exists() else self.ctx.work).free / 1e9
                self.log(f"{self.stage}: {(time.time() - self.t0) / 60:.1f} min in stage, "
                         f"{self.ctx.hours_left():.2f} h of session left | {gpu_line()} | disk free {free:.1f} GB",
                         tag="heartbeat")

    def stop(self):
        self._halt.set()


def run(cmd, log: Log, tag: str, env=None, cwd=None, check=True) -> int:
    """Run a child process, streaming each output line into the cell and the logs as it comes."""
    cmd = [str(c) for c in cmd]
    log(f"$ {' '.join(cmd)}", tag=tag)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})}, cwd=cwd)
    for line in p.stdout:
        line = line.rstrip()
        if line:
            log(line, tag=tag)
    rc = p.wait()
    if rc and check:
        raise RuntimeError(f"{tag}: exit code {rc}")
    return rc


def run_parallel(cmds, log: Log, cwd=None):
    """Several children at once (one per GPU), each line tagged with its child's name."""
    threads, rcs = [], {}

    def go(tag, cmd, env):
        rcs[tag] = run(cmd, log, tag, env=env, cwd=cwd, check=False)

    for tag, cmd, env in cmds:
        th = threading.Thread(target=go, args=(tag, cmd, env))
        th.start()
        threads.append(th)
    for th in threads:
        th.join()
    bad = {k: v for k, v in rcs.items() if v}
    if bad:
        raise RuntimeError(f"failed: {bad}")


# ----------------------------------------------------------------------------- context

class Ctx:
    def __init__(self, a):
        self.a = a
        self.kit = a.kit
        self.work = a.work
        self.reports = a.work / "reports"
        self.state_path = a.work / "state.json"
        self.state = json.loads(self.state_path.read_text()) if self.state_path.exists() else {"stages": {}}
        start_file = a.work / ".session_start"
        if not start_file.exists() or a.new_session:
            start_file.write_text(str(time.time()))
        self.session_start = float(start_file.read_text())
        self.py = sys.executable
        self.n_gpu = self._gpus()

    @staticmethod
    def _gpus() -> int:
        try:
            import torch
            return torch.cuda.device_count()
        except Exception:
            return 0

    def hours_left(self) -> float:
        return self.a.session_hours - (time.time() - self.session_start) / 3600

    def save(self):
        self.state_path.parent.mkdir(parents=True, exist_ok=True)
        self.state_path.write_text(json.dumps(self.state, indent=1, default=str))

    def script(self, rel: str) -> Path:
        p = self.kit / rel
        if not p.exists():
            raise FileNotFoundError(f"{p} missing: run the notebook's %%writefile cells first")
        return p

    @property
    def src(self) -> Path:
        return Path(self.state.get("src", self.a.gsr_input))

    @property
    def dev(self) -> str:
        return "cuda:0" if self.n_gpu else "cpu"

    @property
    def cache(self) -> Path:
        return self.work / "main_cache"

    @property
    def motion(self) -> Path:
        return self.work / "motion_cache"

    @property
    def dataset(self) -> Path:
        return self.work / "c5d_dataset"

    @property
    def c5d_run(self) -> Path:
        return self.work / "runs" / self.a.run_name


# ----------------------------------------------------------------------------- helpers

def find_input(patterns):
    for pat in patterns:
        hits = sorted(KAGGLE_IN.glob(pat))
        if hits:
            return hits[0]
    return None


def tree_bytes(root: Path) -> tuple[int, int]:
    n = b = 0
    for dp, _dn, fn in os.walk(root):
        for f in fn:
            try:
                b += os.path.getsize(os.path.join(dp, f))
                n += 1
            except OSError:
                pass
    return n, b


def copy_tree(src: Path, dst: Path, log: Log, threads: int = 8):
    """Parallel file copy with a progress line every ~10 s (files, GB, MB/s, ETA). Files already
    copied at the right size are skipped, so an interrupted copy resumes."""
    from concurrent.futures import ThreadPoolExecutor
    files = []
    for dp, _dn, fn in os.walk(src):
        for f in fn:
            s = Path(dp) / f
            files.append((s, dst / s.relative_to(src)))
    total = sum(s.stat().st_size for s, _ in files)
    log(f"copying {len(files)} files, {total / 1e9:.1f} GB: {src} -> {dst}", tag="data")
    done = {"n": 0, "b": 0}
    lock = threading.Lock()
    t0, last = time.time(), [0.0]

    def one(pair):
        s, d = pair
        size = s.stat().st_size
        if not (d.exists() and d.stat().st_size == size):
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(s, d)
        with lock:
            done["n"] += 1
            done["b"] += size
            now = time.time()
            if now - last[0] > 10 or done["n"] == len(files):
                last[0] = now
                rate = done["b"] / max(now - t0, 1e-6)
                eta = (total - done["b"]) / max(rate, 1)
                log(f"  {done['n']}/{len(files)} files, {done['b'] / 1e9:.1f}/{total / 1e9:.1f} GB, "
                    f"{rate / 1e6:.0f} MB/s, ETA {eta / 60:.1f} min", tag="data")

    with ThreadPoolExecutor(threads) as ex:
        list(ex.map(one, files))


def sha256(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(1 << 20):
            h.update(b)
    return h.hexdigest()


def pct(v) -> str:
    return "n/a" if v is None else f"{v:.2%}"


# ----------------------------------------------------------------------------- stages

RESTORE_SKIP = {"gsr", "c5d_dataset", ".session_start", "logs"}


def st_restore(ctx: Ctx, log: Log):
    src = ctx.a.restore_from
    if not src:
        log("no --restore_from: nothing to restore (first session)", tag="restore")
        return
    src = Path(src)
    if (src / "ball_improvement").is_dir():
        src = src / "ball_improvement"
    if not (src / "state.json").exists():
        raise FileNotFoundError(f"{src} has no state.json: point --restore_from at an earlier version's "
                                "ball_improvement folder (Add Input -> this notebook's output)")
    for item in sorted(src.iterdir()):
        if item.name in RESTORE_SKIP:
            continue
        dst = ctx.work / item.name
        log(f"restoring {item.name}", tag="restore")
        if item.is_dir():
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            shutil.copyfile(item, dst)
    ctx.state = json.loads((ctx.work / "state.json").read_text())
    ctx.state.pop("src", None)                    # the data stage sets it again for this session
    ctx.state["restored_from"] = str(src)
    ctx.save()
    log(f"restored from {src}; finished stages then: {list(ctx.state.get('stages', {}))}", tag="restore")


def st_env(ctx: Ctx, log: Log):
    if shutil.which("nvidia-smi"):
        run(["nvidia-smi", "-L"], log, "env", check=False)
    if not ctx.a.skip_pip:
        run([ctx.py, "-m", "pip", "install", "-q", "ultralytics>=8.4.0", "onnx", "onnxruntime"], log, "env")
    code = ("import torch, cv2, numpy, ultralytics, onnxruntime; "
            "print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpus', torch.cuda.device_count()); "
            "print('opencv', cv2.__version__, 'numpy', numpy.__version__, 'ultralytics', ultralytics.__version__, "
            "'onnxruntime', onnxruntime.__version__)")
    run([ctx.py, "-c", code], log, "env")
    du = shutil.disk_usage(WORKING)
    log(f"/kaggle/working: {du.free / 1e9:.1f} GB free of {du.total / 1e9:.1f} GB; CPUs {os.cpu_count()}; "
        f"session time left {ctx.hours_left():.2f} h", tag="env")
    if ctx.n_gpu == 0:
        log("WARNING: no GPU visible. Set the notebook accelerator to GPU T4 x2.", tag="env")


def st_data(ctx: Ctx, log: Log):
    a = ctx.a
    gsr = Path(a.gsr_input)
    if not gsr.exists():
        gsr = find_input(["*/soccernetgsr", "datasets/*/soccernetgsr", "*soccernet*"])
        if gsr is None:
            raise FileNotFoundError(f"GSR input not found at {a.gsr_input}; add it (Add Input) or pass --gsr_input")
    log(f"GSR input: {gsr}", tag="data")

    wdir = ctx.work / "weights"
    wdir.mkdir(parents=True, exist_ok=True)
    main_w = Path(a.main_weights) if a.main_weights else find_input(
        ["*/ball-detector-scripts/best.pt", "datasets/*/ball-detector-scripts/best.pt"])
    c5c = Path(a.c5c_ckpt) if a.c5c_ckpt else find_input(
        ["**/c5c_best.pt", "**/ball_temporal/c5c/best.pt", "**/c5c/best.pt"])
    if main_w is None or not main_w.exists():
        raise FileNotFoundError("main detector best.pt not found under /kaggle/input; pass --main_weights")
    if c5c is None or not c5c.exists():
        raise FileNotFoundError("c5c checkpoint not found: upload runs/ball_temporal/c5c/best.pt as c5c_best.pt "
                                "in a dataset (see the notebook's first cell) or pass --c5c_ckpt")
    shutil.copyfile(main_w, wdir / "main_best.pt")
    shutil.copyfile(c5c, wdir / "c5c_best.pt")
    log(f"weights: main {main_w} ({(wdir / 'main_best.pt').stat().st_size / 1e6:.0f} MB), "
        f"c5c {c5c} ({(wdir / 'c5c_best.pt').stat().st_size / 1e6:.0f} MB) -> {wdir}", tag="data")

    n, b = tree_bytes(gsr)
    free = shutil.disk_usage(WORKING).free
    need = b + a.reserve_gb * 1e9
    log(f"GSR input: {n} files, {b / 1e9:.1f} GB; /kaggle/working free {free / 1e9:.1f} GB; copying needs "
        f"{need / 1e9:.1f} GB ({a.reserve_gb} GB kept for caches and the crop dataset)", tag="data")
    dst = ctx.work / "gsr"
    if a.copy == "never" or (a.copy == "auto" and free < need):
        why = "--copy never" if a.copy == "never" else "not enough space in /kaggle/working"
        log(f"NOT copying ({why}): reading the dataset in place from {gsr}; every stage only reads it. "
            f"--copy always forces the copy; a lower --reserve_gb makes room.", tag="data")
        ctx.state["src"] = str(gsr)
    else:
        copy_tree(gsr, dst, log, threads=a.copy_threads)
        ctx.state["src"] = str(dst)
    ctx.save()
    sys.path.insert(0, str(ctx.kit / "scripts" / "ball_temporal"))
    from gsr_io import discover
    refs = discover(ctx.src)
    counts = {s: sum(r.split == s for r in refs) for s in ("train", "pool", "select", "test")}
    log(f"splits in {ctx.src}: {counts}", tag="data")
    if a.expect_test and counts["test"] != a.expect_test:
        raise RuntimeError(f"expected {a.expect_test} test sequences, found {counts['test']}: the valid split is incomplete")
    if counts["select"] != 10:
        log(f"WARNING: {counts['select']} select sequences (expected 10)", tag="data")
    ctx.state["splits"] = counts
    ctx.save()


def st_cache(ctx: Ctx, log: Log):
    a = ctx.a
    if a.main_cache_zip:
        log(f"unzipping an earlier main cache: {a.main_cache_zip}", tag="cache")
        with zipfile.ZipFile(a.main_cache_zip) as zf:
            zf.extractall(ctx.work)
    script = ctx.script("scripts/ball_temporal/cache_main_detections.py")
    base = [ctx.py, script, "run", "--src", ctx.src, "--out", ctx.cache, "--weights", ctx.work / "weights" / "main_best.pt",
            "--imgsz", "1280"]
    if a.max_frames:
        base += ["--max_frames", str(a.max_frames)]
    if ctx.n_gpu <= 1:
        run(base + (["--device", "0", "--half"] if ctx.n_gpu else ["--device", "cpu"]), log, "cache")
    else:
        run_parallel([(f"cache-gpu{i}", base + ["--device", str(i), "--half", "--shard", str(i),
                                                "--nshards", str(ctx.n_gpu)], {}) for i in range(ctx.n_gpu)], log)
    run([ctx.py, script, "summary", "--out", ctx.cache], log, "cache")
    if (ctx.cache / "summary.md").exists():
        shutil.copyfile(ctx.cache / "summary.md", ctx.reports / "main_cache_summary.md")


def _motion(ctx: Ctx, log: Log, splits):
    script = ctx.script("scripts/ball_temporal/camera_motion_cache.py")
    run([ctx.py, script, "run", "--src", ctx.src, "--out", ctx.motion, "--splits", *splits,
         "--workers", str(os.cpu_count() or 4)], log, "motion")
    run([ctx.py, script, "summary", "--out", ctx.motion], log, "motion")


def st_motion(ctx: Ctx, log: Log):
    _motion(ctx, log, ["select", "test"])


def st_diagnose(ctx: Ctx, log: Log):
    out = ctx.reports / "diagnosis"
    cmd = [ctx.py, ctx.script("scripts/eval/ball_det/diagnose_ball_misses.py"), "run", "--src", ctx.src,
           "--cache", ctx.cache, "--motion", ctx.motion, "--ckpt", ctx.work / "weights" / "c5c_best.pt",
           "--out", out, "--device", ctx.dev]
    if ctx.a.max_seqs:
        cmd += ["--max_seqs", str(ctx.a.max_seqs)]
    run(cmd, log, "diagnose")
    d = json.loads((out / "diagnosis.json").read_text())
    ctx.state["diagnosis"] = {s: {k: v[k] for k in ("recall", "ball_frames", "misses", "primary", "ceiling")}
                              for s, v in d["splits"].items()}
    ctx.state["gate_fit"] = d.get("gate_fit")
    ctx.save()
    c = d["splits"].get("test", {}).get("ceiling")
    if c:
        log(f"test: recall now {c['now']:.1%}; live ceiling {c['live_ceiling']:.1%}, batch ceiling "
            f"{c['batch_ceiling']:.1%} (item 2 +{c['item2']:.1%}, item 3 +{c['item3']:.1%}, "
            f"item 4 +{c['item4']:.1%}, hidden behind players {c['occluded']:.1%})", tag="diagnose")


def _track(ctx: Ctx, log: Log, ckpt: Path, out: Path, tag: str) -> dict:
    cmd = [ctx.py, ctx.script("scripts/eval/ball_det/eval_ball_temporal.py"), "track", "--ckpt", ckpt,
           "--src", ctx.src, "--cache", ctx.cache, "--motion", ctx.motion, "--out", out,
           "--device", ctx.dev, "--search_passes", str(ctx.a.search_passes)]
    if ctx.a.max_seqs:
        cmd += ["--max_seqs", str(ctx.a.max_seqs)]
    run(cmd, log, tag)
    return json.loads((out / "results_track.json").read_text())


def st_track(ctx: Ctx, log: Log):
    r = _track(ctx, log, ctx.work / "weights" / "c5c_best.pt", ctx.reports / "track_c5c", "track")
    sim = r["config"]["sim"]
    ctx.state["track_c5c"] = {"gate_t": r["gate_t"], "live": r["test"]["live"]["recall"],
                              "live_precision": r["test"]["live"]["precision"],
                              "batch_interp": r["test"]["batch_interp"]["recall"],
                              "c5c": r["test_c5c"]["recall"], "main": r["test_main"]["recall"], "sim": sim}
    ctx.save()
    log(f"gate T {'PASS' if r['gate_t'] else 'FAIL'}: live recall {r['test']['live']['recall']:.2%} "
        f"(c5c as wired {r['test_c5c']['recall']:.2%}), batch+interp {r['test']['batch_interp']['recall']:.2%}; "
        f"window centre {sim['window_centre']}, {sim['n_windows']} window(s)", tag="track")


def st_c5d_data(ctx: Ctx, log: Log):
    a = ctx.a
    sim = ctx.state.get("track_c5c", {}).get("sim", {})
    centre = a.centre if a.centre != "auto" else sim.get("window_centre", "last_seen")
    log(f"c5d window centre: {centre} ({'chosen by the track stage' if a.centre == 'auto' else '--centre'})",
        tag="c5d_data")
    if centre == "predicted":
        _motion(ctx, log, ["train", "pool", "select", "test"])
    if (ctx.dataset / "manifest.csv").exists():
        log("dataset exists (manifest.csv): skipping the build; delete the folder to rebuild", tag="c5d_data")
    else:
        cmd = [ctx.py, ctx.script("scripts/ball_temporal/build_ball_temporal_dataset.py"), "build", "--src", ctx.src,
               "--cache", ctx.cache, "--out", ctx.dataset, "--workers", str(os.cpu_count() or 4),
               "--centre", centre, "--gap2_repeat", str(a.gap2_repeat), "--jpeg_quality", str(a.jpeg_quality)]
        if centre == "predicted":
            cmd += ["--motion", ctx.motion, "--tracker_config", ctx.reports / "track_c5c" / "tracker_config.json"]
        run(cmd, log, "c5d_data")
    shutil.copyfile(ctx.dataset / "stats.md", ctx.reports / "c5d_dataset_stats.md")
    ctx.state["c5d_centre"] = centre
    ctx.save()
    _, b = tree_bytes(ctx.dataset)
    log(f"crop dataset: {b / 1e9:.1f} GB at {ctx.dataset}", tag="c5d_data")


def st_c5d_train(ctx: Ctx, log: Log):
    a = ctx.a
    out = ctx.c5d_run
    out.mkdir(parents=True, exist_ok=True)
    if (out / "DONE").exists():
        log(f"{out} already finished (DONE); skipping", tag="c5d_train")
        return
    budget = ctx.hours_left() - a.reserve_hours
    if budget < 0.5:
        raise RuntimeError(f"only {ctx.hours_left():.2f} h of session left with {a.reserve_hours} h reserved for "
                           "eval and export: Save Version, start a new session and re-run (training resumes)")
    nproc = max(1, min(ctx.n_gpu, a.gpus))
    args = ["--data", ctx.dataset, "--out", out, "--channels", "5", "--motion_blur", "0.25",
            "--epochs", str(a.epochs), "--lr", str(a.lr), "--batch", str(a.batch), "--workers", str(a.workers),
            "--patience", str(a.patience), "--time_budget_h", f"{budget:.2f}",
            "--init", ctx.work / "weights" / "c5c_best.pt", "--no_pretrained"]
    if (out / "last.pt").exists():
        args.append("--resume")
        log("last.pt found: resuming", tag="c5d_train")
    if a.limit:
        args += ["--limit", str(a.limit)]
    script = ctx.script("scripts/ball_temporal/train_ball_temporal.py")
    log(f"training {a.run_name}: {nproc} GPU(s), up to {a.epochs} epochs, lr {a.lr}, time budget {budget:.2f} h",
        tag="c5d_train")
    if nproc > 1:
        run([ctx.py, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={nproc}", script, *args],
            log, "c5d_train")
    else:
        run([ctx.py, script, *args, "--device", ctx.dev], log, "c5d_train")
    if (out / "INCOMPLETE").exists():
        ctx.state["c5d_train"] = "incomplete"
        ctx.save()
        raise RuntimeError("training stopped at the session time budget (INCOMPLETE). Run the cleanup cell, Save "
                           "Version, then in a new session add this notebook's output as an input and run the "
                           "resume cell (restore -> data -> c5d_data -> c5d_train); it continues from last.pt.")
    ctx.state["c5d_train"] = json.loads((out / "DONE").read_text())
    ctx.save()


def st_c5d_eval(ctx: Ctx, log: Log):
    ckpt = ctx.c5d_run / "best.pt"
    out = ctx.reports / f"eval_{ctx.a.run_name}"
    run([ctx.py, ctx.script("scripts/eval/ball_det/eval_ball_temporal.py"), "all", "--ckpt", ckpt, "--src", ctx.src,
         "--cache", ctx.cache, "--data", ctx.dataset, "--out", out, "--device", ctx.dev,
         "--workers", str(ctx.a.workers)], log, "c5d_eval")
    r = json.loads((out / "results.json").read_text())
    thr = float(r["system_thr"])
    ctx.state["c5d_gates"] = {"gate1": r["gate1"], "gate3": r["gate3"], "system_thr": thr,
                              "recall": r["test"]["recall"], "precision": r["test"]["precision"],
                              "main_recall": r["test_main"]["recall"]}
    ctx.save()
    log(f"c5d: gate 1 {'PASS' if r['gate1'] else 'FAIL'}, gate 3 {'PASS' if r['gate3'] else 'FAIL'}; "
        f"test recall {r['test']['recall']:.2%} at threshold {thr:.2f}", tag="c5d_eval")
    t = _track(ctx, log, ckpt, ctx.reports / f"track_{ctx.a.run_name}", "c5d_track")
    gap25 = [v for k, v in t["test"]["live"]["by_gap"].items() if k in ("2", "3", "4-5")]
    ctx.state["track_c5d"] = {"gate_t": t["gate_t"], "live": t["test"]["live"]["recall"],
                              "live_precision": t["test"]["live"]["precision"],
                              "batch_interp": t["test"]["batch_interp"]["recall"],
                              "gap2_5": (sum(r_ * n for r_, n in gap25) / max(sum(n for _, n in gap25), 1)) if gap25 else None}
    ctx.save()
    log(f"c5d + tracker: live recall {t['test']['live']['recall']:.2%} (c5c + tracker "
        f"{pct(ctx.state.get('track_c5c', {}).get('live'))}), batch+interp {t['test']['batch_interp']['recall']:.2%}",
        tag="c5d_eval")


def st_export(ctx: Ctx, log: Log):
    thr = ctx.state.get("c5d_gates", {}).get("system_thr")
    if thr is None:
        raise RuntimeError("run c5d_eval first: it chooses the threshold the sidecar records")
    script = ctx.script("scripts/ball_temporal/export_ball_specialist.py")
    batches = [1] + ([3] if ctx.state.get("track_c5c", {}).get("sim", {}).get("n_windows") == 3 else [])
    for b in batches:
        name = f"ball_specialist_{ctx.a.run_name}" + ("" if b == 1 else f"_b{b}") + ".onnx"
        run([ctx.py, script, "--ckpt", ctx.c5d_run / "best.pt", "--out", ctx.c5d_run / name, "--threshold", str(thr),
             "--data", ctx.dataset, "--batch", str(b), "--report", ctx.reports / f"parity_b{b}.json"], log, "export")
    ctx.state["export"] = {"batches": batches, "threshold": thr}
    ctx.save()


def _summary(ctx: Ctx) -> str:
    s = ctx.state
    d = s.get("diagnosis", {}).get("test", {})
    c = d.get("ceiling", {})
    tc, td, g = s.get("track_c5c", {}), s.get("track_c5d", {}), s.get("c5d_gates", {})

    def gate(v):
        return "not run" if v is None else ("PASS" if v else "FAIL")

    L = [f"# Ball improvement run, {datetime.now():%Y-%m-%d %H:%M}", "",
         "Plan: `docs/plans/BALL_IMPROVEMENT_PLAN.md` items 1-3. Test = the 15 held-out sequences, 15 px centre "
         "match. Every setting was chosen on `select`.", "",
         "| | test ball recall | precision |", "|---|---|---|",
         f"| main detector alone | {pct(tc.get('main'))} | |",
         f"| main + c5c, as wired today | {pct(tc.get('c5c'))} | |",
         f"| c5c + tracker, live (measured + coasted) | {pct(tc.get('live'))} | {pct(tc.get('live_precision'))} |",
         f"| c5c + tracker, batch path + interpolation | {pct(tc.get('batch_interp'))} | |",
         f"| c5d + tracker, live | {pct(td.get('live'))} | {pct(td.get('live_precision'))} |",
         f"| c5d + tracker, batch path + interpolation | {pct(td.get('batch_interp'))} | |", "",
         "## Gates", "",
         f"- Gate T (tracker beats c5c as wired, live): **{gate(tc.get('gate_t'))}**",
         f"- c5d gate 1: **{gate(g.get('gate1'))}**, gate 3: **{gate(g.get('gate3'))}** "
         f"(threshold {g.get('system_thr', 'n/a')})",
         f"- Gate S (c5d + tracker, recall at gap 2-5 at least double c5c's 18%): "
         f"**{gate(None if td.get('gap2_5') is None else td['gap2_5'] >= 0.36)}** (gap 2-5 recall {pct(td.get('gap2_5'))})",
         "", "## Diagnosis (test)", ""]
    if c:
        L += [f"- Live ceiling {c['live_ceiling']:.1%}, batch ceiling {c['batch_ceiling']:.1%} (now {c['now']:.1%}).",
              f"- Addressable: item 2 +{c['item2']:.1%}, item 3 +{c['item3']:.1%}, item 4 +{c['item4']:.1%}, "
              f"localisation +{c['near_miss']:.1%}; hidden behind players {c['occluded']:.1%}.",
              f"- Primary causes (counts): {d.get('primary')}", "",
              "Decide item 4 (detector retrain) from these numbers, as the plan requires (§1)."]
    else:
        L.append("- not run")
    L += ["", "## Stages", ""] + [f"- {k}: {v}" for k, v in s.get("stages", {}).items()]
    L += ["", "## Next", "",
          "1. Read `reports/diagnosis/report_diagnosis.md`; do the audit pass (`reports/diagnosis/audit/`).",
          "2. Gate T passed: wire `gsr_pipeline/ball_tracker.py` into the pipeline with `configs/tracker_config.json`.",
          "3. c5d passed its gates: copy `models/` to `runs/ball_temporal/<run>/` and point "
          "`ball_specialist.onnx_path` at the batch-1 ONNX.",
          "4. Paste `PERF_LOG_entry.md` into `docs/path_b/PERF_LOG.md`."]
    return "\n".join(L) + "\n"


def _perf_log(ctx: Ctx) -> str:
    s = ctx.state
    tc, td, g = s.get("track_c5c", {}), s.get("track_c5d", {}), s.get("c5d_gates", {})
    c = s.get("diagnosis", {}).get("test", {}).get("ceiling", {})
    return "\n".join([
        f"## {datetime.now():%Y-%m-%d}: ball improvement plan items 1-3 (Kaggle)", "",
        f"Tag: `kaggle-t4` ({ctx.n_gpu} GPU). Run `{ctx.a.run_name}`. Reports: the run's deliverables.", "",
        f"- **Diagnosis (test):** recall {pct(c.get('now'))}; live ceiling {pct(c.get('live_ceiling'))}, "
        f"batch ceiling {pct(c.get('batch_ceiling'))}.",
        f"- **Tracker (c5c):** live {pct(tc.get('live'))} at precision {pct(tc.get('live_precision'))} vs as wired "
        f"{pct(tc.get('c5c'))}; batch + interpolation {pct(tc.get('batch_interp'))}. Gate T: {tc.get('gate_t')}. "
        f"Chosen: {tc.get('sim')}.",
        f"- **{ctx.a.run_name}:** gate 1 {g.get('gate1')}, gate 3 {g.get('gate3')} at threshold {g.get('system_thr')}; "
        f"with the tracker live {pct(td.get('live'))}, batch + interpolation {pct(td.get('batch_interp'))}, "
        f"gap 2-5 {pct(td.get('gap2_5'))}.", ""])


def st_deliver(ctx: Ctx, log: Log):
    d = WORKING / "deliverables"
    if d.exists():
        shutil.rmtree(d)
    (d / "reports").mkdir(parents=True)
    if ctx.reports.exists():
        shutil.copytree(ctx.reports, d / "reports", dirs_exist_ok=True)
    cfg = ctx.reports / "track_c5c" / "tracker_config.json"
    if cfg.exists():
        (d / "configs").mkdir()
        shutil.copyfile(cfg, d / "configs" / "tracker_config.json")
    if ctx.c5d_run.exists():
        (d / "models").mkdir()
        for name in ("best.pt", "history.csv", "curves.png", "args.json", "DONE", "INCOMPLETE"):
            if (ctx.c5d_run / name).exists():
                shutil.copyfile(ctx.c5d_run / name, d / "models" / name)
        for p in ctx.c5d_run.glob("*.onnx*"):
            shutil.copyfile(p, d / "models" / p.name)
    if ctx.cache.exists():
        log("zipping the main cache (reusable next session: --main_cache_zip)", tag="deliver")
        shutil.make_archive(str(d / "main_cache"), "zip", ctx.work, "main_cache")
    if (ctx.work / "logs").exists():
        shutil.copytree(ctx.work / "logs", d / "logs", dirs_exist_ok=True)
    (d / "state.json").write_text(json.dumps(ctx.state, indent=1, default=str))
    (d / "SUMMARY.md").write_text(_summary(ctx), encoding="utf-8")
    (d / "PERF_LOG_entry.md").write_text(_perf_log(ctx), encoding="utf-8")
    manifest = {str(p.relative_to(d)).replace("\\", "/"): {"bytes": p.stat().st_size, "sha256": sha256(p)}
                for p in sorted(d.rglob("*")) if p.is_file()}
    (d / "MANIFEST.json").write_text(json.dumps(manifest, indent=1))
    z = WORKING / "deliverables.zip"
    z.unlink(missing_ok=True)
    shutil.make_archive(str(z.with_suffix("")), "zip", WORKING, "deliverables")
    log(f"deliverables: {d} ({len(manifest)} files), {z} ({z.stat().st_size / 1e6:.0f} MB)", tag="deliver")
    print("\n" + (d / "SUMMARY.md").read_text(encoding="utf-8"), flush=True)


def st_cleanup(ctx: Ctx, log: Log):
    for p in (ctx.work / "gsr", ctx.dataset):
        if p.exists():
            _, b = tree_bytes(p)
            shutil.rmtree(p)
            log(f"deleted {p} ({b / 1e9:.1f} GB)", tag="cleanup")
    last = ctx.c5d_run / "last.pt"
    if last.exists() and (ctx.c5d_run / "DONE").exists():
        last.unlink()
        log(f"deleted {last} (training finished; best.pt is kept)", tag="cleanup")
    log(f"/kaggle/working now {tree_bytes(WORKING)[1] / 1e9:.1f} GB (Save Version keeps up to 20 GB)", tag="cleanup")


FUNCS = {"restore": st_restore, "env": st_env, "data": st_data, "cache": st_cache, "motion": st_motion, "diagnose": st_diagnose,
         "track": st_track, "c5d_data": st_c5d_data, "c5d_train": st_c5d_train, "c5d_eval": st_c5d_eval,
         "export": st_export, "deliver": st_deliver, "cleanup": st_cleanup}
ESTIMATE = {"restore": "~1-5 min", "env": "1-3 min", "data": "~2-4 min per 10 GB copied", "cache": "~30-60 min on 2xT4 (all splits)",
            "motion": "~5-10 min", "diagnose": "~10-20 min", "track": "~1-2 h (search on select)",
            "c5d_data": "~30-60 min (+ motion for train/pool if predicted)", "c5d_train": "up to the session budget",
            "c5d_eval": "~1-2 h", "export": "~5 min", "deliver": "~2-5 min", "cleanup": "~1 min"}


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--stages", default="all", help=f"comma list from {STAGES}, or all")
    ap.add_argument("--kit", type=Path, default=WORKING / "kit" / "cv_service", help="where the %%writefile cells put the code")
    ap.add_argument("--work", type=Path, default=WORKING / "ball_improvement")
    ap.add_argument("--gsr_input", default=str(KAGGLE_IN / "datasets" / "sabihalii" / "soccernetgsr"))
    ap.add_argument("--main_weights", default=None, help="main detector best.pt (default: found under /kaggle/input)")
    ap.add_argument("--c5c_ckpt", default=None, help="c5c best.pt (default: **/c5c_best.pt under /kaggle/input)")
    ap.add_argument("--main_cache_zip", default=None, help="main_cache.zip from an earlier run: skip recomputing")
    ap.add_argument("--restore_from", default=None,
                    help="restore stage: an earlier version's output folder under /kaggle/input (its ball_improvement/)")
    ap.add_argument("--copy", choices=["auto", "always", "never"], default="auto")
    ap.add_argument("--copy_threads", type=int, default=8)
    ap.add_argument("--reserve_gb", type=float, default=25.0, help="space kept for caches + crop dataset when copying")
    ap.add_argument("--session_hours", type=float, default=12.0)
    ap.add_argument("--new_session", action="store_true", help="reset the session clock (first cell of a new session)")
    ap.add_argument("--reserve_hours", type=float, default=2.5, help="session time kept after training for eval/export/deliver")
    ap.add_argument("--run_name", default="c5d")
    ap.add_argument("--centre", choices=["auto", "last_seen", "predicted"], default="auto")
    ap.add_argument("--gap2_repeat", type=int, default=2)
    ap.add_argument("--jpeg_quality", type=int, default=92)
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--lr", type=float, default=1.5e-4, help="warm start: ~0.3x the original 5e-4")
    ap.add_argument("--batch", type=int, default=32, help="per GPU")
    ap.add_argument("--gpus", type=int, default=2)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--patience", type=int, default=6)
    ap.add_argument("--search_passes", type=int, default=2)
    ap.add_argument("--max_seqs", type=int, default=0, help="smoke tests: first N sequences per split (diagnose/track)")
    ap.add_argument("--max_frames", type=int, default=0, help="smoke tests: first N frames per sequence (cache)")
    ap.add_argument("--limit", type=int, default=0, help="smoke tests: samples per epoch (train)")
    ap.add_argument("--heartbeat_s", type=int, default=60)
    ap.add_argument("--skip_pip", action="store_true", help="env: don't pip-install (packages already present)")
    ap.add_argument("--expect_test", type=int, default=15, help="data: required number of test sequences (0 = any)")
    a = ap.parse_args()

    stages = STAGES if a.stages == "all" else [s.strip() for s in a.stages.split(",") if s.strip()]
    unknown = [s for s in stages if s not in FUNCS]
    if unknown:
        raise SystemExit(f"unknown stages {unknown}; choose from {STAGES}")
    a.work.mkdir(parents=True, exist_ok=True)
    log = Log(a.work / "logs")
    ctx = Ctx(a)
    ctx.reports.mkdir(parents=True, exist_ok=True)
    hb = Heartbeat(log, ctx, a.heartbeat_s)
    hb.start()
    log(f"stages {stages}; kit {a.kit}; work {a.work}; {ctx.n_gpu} GPU(s); {ctx.hours_left():.2f} h of session left")
    failed = None
    try:
        for s in stages:
            log.stage(s)
            log.banner(f"stage {s}  (typical: {ESTIMATE[s]})  {ctx.hours_left():.2f} h of session left")
            hb.set(s)
            t0 = time.time()
            try:
                FUNCS[s](ctx, log)
                ctx.state["stages"][s] = f"done in {(time.time() - t0) / 60:.1f} min ({datetime.now():%H:%M})"
                ctx.save()
                log(f"stage {s} done in {(time.time() - t0) / 60:.1f} min")
            except Exception as exc:
                ctx.state["stages"][s] = f"FAILED after {(time.time() - t0) / 60:.1f} min: {exc}"
                ctx.save()
                log(f"stage {s} FAILED: {exc}")
                failed = s
                break
    finally:
        hb.set(None)
        if failed and failed != "deliver":
            log.stage("deliver")
            log("packaging what exists so far, so this session's results are not lost")
            try:
                st_deliver(ctx, log)
            except Exception as exc:
                log(f"deliver also failed: {exc}")
        hb.stop()
        log.stage(None)
    if failed:
        raise SystemExit(f"stopped at stage {failed}; see {a.work / 'logs' / (failed + '.log')}. Re-run to resume.")


if __name__ == "__main__":
    main()

## 1. Setup: environment and data

Installs `ultralytics>=8.4`, `onnx`, `onnxruntime`; copies the weights, and the GSR dataset if it fits; checks the splits (15 test sequences required).

In [ ]:
RUN("env,data")

## 2. Main-detector cache and camera motion

Up to 5 ball candidates per frame at conf >= 0.05, every split, one process per GPU (lines tagged `cache-gpu0` / `cache-gpu1`). Then camera motion for `select` and `test`. ~30-60 min.

In [ ]:
RUN("cache,motion")
SHOW("reports/main_cache_summary.md")

## 3. Item 1: miss diagnosis

Replays main + c5c as wired, tags every miss by cause, measures ball displacement, estimates the ceiling per item, and writes 200 audit sheets (`reports/diagnosis/audit/`).

In [ ]:
RUN("diagnose")
SHOW("reports/diagnosis/report_diagnosis.md")

## 4. Item 2: ball tracker (+ item 3 window variants)

Coordinate search on `select` (tracker knobs, then window centre / 3 windows / run-below, then the batch path), frozen and reported on `test`. Gate T at the end. ~1-2 h.

In [ ]:
RUN("track")
SHOW("reports/track_c5c/report_track.md")

## 5. Item 3: c5d dataset

Window centre as chosen in step 4; pool hard positives >= 2 frames from the last sighting repeated x2.

In [ ]:
RUN("c5d_data")
SHOW("reports/c5d_dataset_stats.md")

## 6. Item 3: c5d training

2-GPU DDP via torchrun, warm start from c5c. Per-100-step lines with loss, lr, s/it, epoch ETA and GPU memory; per-epoch `select` score and run ETA. Stops before the session budget runs out.

In [ ]:
RUN("c5d_train")

## 7. Item 3: c5d gates and export

Gates 1-3 with a new threshold chosen on `select`, the tracker re-run with c5d (gate S), then the ONNX export and parity check.

In [ ]:
RUN("c5d_eval,export")
SHOW("reports/eval_c5d/report.md")
SHOW("reports/track_c5d/report_track.md")

## 8. Deliverables

`/kaggle/working/deliverables/`: `SUMMARY.md`, `PERF_LOG_entry.md`, `reports/`, `configs/tracker_config.json`, `models/` (best.pt, ONNX + sidecar, history, curves), `main_cache.zip`, `logs/`, `MANIFEST.json` (sha256 per file); and `deliverables.zip`. A failed stage packages what exists automatically.

In [ ]:
RUN("deliver")
from IPython.display import Markdown, display
display(Markdown(Path("/kaggle/working/deliverables/SUMMARY.md").read_text(encoding="utf-8")))

## 9. Cleanup before Save Version

Deletes the GSR copy and the crop dataset (both rebuilt on resume) so the output fits in 20 GB.

In [ ]:
RUN("cleanup")

## Resume in a new session (only if training stopped at the time budget)

1. In the finished session: run **Cleanup**, then **Save Version** (Quick Save).
2. New session: Add Input -> this notebook's output. Run the Settings cell and the kit cells.
3. Set `PREV` below to that input's `ball_improvement` folder and run the cell. It restores the state,
   caches, reports and `last.pt`, rebuilds the crop dataset, and continues training. Then run cells 7-9.

In [ ]:
PREV = "/kaggle/input/<this-notebook-output>/ball_improvement"   # edit
RUN("restore,env,data,c5d_data,c5d_train", "--restore_from", PREV)